# D2 · la estrella primaria — notebook de análisis (`debug`)

**Objeto:** ROXs12b  |  **Run:** `ROXs12b_realigned`  |  **Spec:** [`docs/spec_D2_codex_spectral_calibration.md`](../../../docs/spec_D2_codex_spectral_calibration.md)

Los otros notebooks de análisis van sobre el **compañero**: la fuente débil que hay que sacar de debajo del halo. Éste va sobre la **estrella central**, que es el otro espectro que la cadena entrega — C4 la extrae junto al compañero (`spec_psffit_star.fits`) y D2 la calibra con el mismo camino (`spec_calibrated_psffit_star.fits`).

Y trae una pregunta que **ninguna etapa hace**: cómo es el espectro de la primaria **en cada observación por separado**. La cadena entera trabaja sobre un solo cubo, el combinado, y ahí ya no se puede distinguir una noche mala de una buena, ni ver si la estrella cambió entre exposiciones. La §9 lo mide, exposición a exposición.

Cómo está montado, y por qué:

1. **Perillas** arriba del todo, con el valor que usa la cadena para este run.
2. **Las funciones numéricas, copiadas literalmente** de `musepipe`: la calibración de D2 entera, más la extracción por apertura que usa A3 (la §9 mide con **esa** y no con una reimplementación).
3. **Chequeo de deriva** — avisa si `musepipe` cambió y esta copia se quedó atrás.
4. La calibración **paso a paso**, cada corrección con su origen.
5. **Comparación con el producto de la cadena**: con las perillas por defecto debe salir *idéntico*.
6. **El espectro de cada observación** (§9), que es lo que no está en ningún QC.

> Por qué importa la primaria y no es un adorno: **es la fuente del halo**. Todo lo que las etapas C restan —anillo, modelo de PSF, referencia estelar— sale de ella. Su espectro es el del contaminante, su brillo fija el contraste y su barra de error manda en el presupuesto del compañero. Y es la única fuente con S/N suficiente para medir en una sola exposición.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

# Resolución de las figuras EN PANTALLA (ver el mismo bloque en C2).
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
# El objeto se DERIVA del run, no se escribe (tests/test_no_hardcoded_target.py).
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)
print('stages :', SD)


## 1 · Perillas

Las de la calibración salen del **config resuelto de la etapa**, no del `config.json` crudo: D2 rellena defaults que el run no escribe, y copiarlos a mano es exactamente cómo se consigue un notebook que no reproduce la cadena.

Las de la §9 **no son de ninguna etapa** —no hay etapa que mida por exposición— así que se declaran aquí, con una excepción: el **radio de apertura** se toma del que usó A3 para medir la primaria en su QC, para que la medida por exposición y la de la cadena sean la misma medida.


In [ ]:
from musepipe.stages.stage_x11_calibrate import (
    stage_x11_config_from_run, stage_x11_paths)

# `project_root=ROOT` no es opcional: musepipe resuelve rutas contra el cwd, y
# el cwd de un notebook es su propia carpeta, no la raíz del repo.
X11 = stage_x11_config_from_run(RUN_ID, project_root=ROOT)   # run + defaults de la etapa
CONT_WINDOW_A = float(X11.get('x11_continuum_window_A', 80.0))
CONT_POLY_DEG = int(X11.get('x11_continuum_poly_deg', 5))
ERR_SMOOTH_CH = int(X11.get('x11_error_smooth_channels', 21))

# ---- perillas de la §9 (el espectro por observación) ----
# El radio sale del QC de A3: es el que la cadena usó para medir esta misma
# estrella. Si A3 no ha corrido para este objeto, 8 px (0.2" en NFM).
_QC_A3 = nb.load_qc_optional('stages/stage00t_qc.json', RUN_ID) or {}
RADIO_PX = float((_QC_A3.get('input') or {}).get('aperture_radius_px') or 8.0)
RADIOS_PX = (2.0, 4.0, 8.0, 16.0, 24.0, 32.0)   # la curva de crecimiento
VENTANA_PX = 35        # media caja leída del cubo alrededor de la estrella
PASO_BLANCO = 37       # 1 de cada N canales para la imagen blanca (localización)
BANDA_LIMPIA_A = (7750.0, 7860.0)   # sin telúrico ni líneas: donde se normaliza
HALPHA_A = (6540.0, 6590.0)
HALPHA_LATERAL_A = ((6470.0, 6530.0), (6600.0, 6660.0))

# ---- a partir de aquí, cambia lo que quieras probar ----

for _k, _v in {'ventana continuo [Å]': CONT_WINDOW_A, 'grado polinomio': CONT_POLY_DEG,
               'suavizado error [ch]': ERR_SMOOTH_CH,
               'radio apertura [px]': RADIO_PX,
               'banda limpia [Å]': BANDA_LIMPIA_A}.items():
    print(f'  {_k:22s} {_v}')


## 2 · Entradas

Las mismas que toma D2 para la primaria, y **de dónde sale cada una**. Son pocas y conviene verlas juntas: el espectro que escribió C4, y los cuatro QC que fijan las correcciones (λ y escala de flujo de A4, PSF de C1, cielo de A2, telúrico de A3).

La celda comprueba además una cosa que no comprueba nadie: que el producto calibrado en disco sea **más nuevo que su entrada**. Si C4 se re-ejecutó después de D2, lo que hay en `stages/` es un calibrado de un espectro que ya no existe, y la comparación de la §7 saldrá `DIFIERE` por esa razón y no porque la copia esté mal.


In [ ]:
PATHS = stage_x11_paths(RUN_ID, ROOT)

def _json_opcional(ruta):
    ruta = Path(ruta)
    return json.loads(ruta.read_text(encoding='utf-8')) if ruta.exists() else None

QC00 = _json_opcional(X11.get('x11_stage00q_qc_json', PATHS['stage00q_qc_json']))
QC_SKY = _json_opcional(X11.get('x11_stage00s_qc_json', PATHS['stage00s_qc_json']))
QC_TEL = _json_opcional(X11.get('x11_stage00t_qc_json', PATHS['stage00t_qc_json']))
QC_PSF = _json_opcional(X11.get('x11_stage_e01_qc_json', PATHS['stage_e01_qc_json']))
ENTRADA_C4 = Path(X11.get('x11_spec_psffit_star', PATHS['spec_psffit_star']))
SALIDA_D2 = Path(PATHS['spec_calibrated_psffit_star'])

print('QC leídos:')
for _n, _q, _p in (('A4 (λ y flujo)', QC00, PATHS['stage00q_qc_json']),
                   ('A2 (cielo)', QC_SKY, PATHS['stage00s_qc_json']),
                   ('A3 (telúrico)', QC_TEL, PATHS['stage00t_qc_json']),
                   ('C1 (PSF)', QC_PSF, PATHS['stage_e01_qc_json'])):
    print(f'  {_n:16s} {"sí" if _q else "NO ESTÁ":8s} {Path(_p).name}')

print('\nproductos:')
import datetime as _dt
def _cuando(p):
    p = Path(p)
    if not p.exists():
        return None, 'no está'
    t = p.stat().st_mtime
    return t, _dt.datetime.fromtimestamp(t).strftime('%Y-%m-%d %H:%M')
_t_c4, _s_c4 = _cuando(ENTRADA_C4)
_t_d2, _s_d2 = _cuando(SALIDA_D2)
print(f'  C4 (entrada)  {ENTRADA_C4.name:34s} {_s_c4}')
print(f'  D2 (salida)   {SALIDA_D2.name:34s} {_s_d2}')
# El producto de D2 más viejo que su entrada NO es un detalle: significa que
# la primaria calibrada del run corresponde a otra extracción.
D2_AL_DIA = bool(_t_c4 and _t_d2 and _t_d2 >= _t_c4)
if _t_c4 and _t_d2 and not D2_AL_DIA:
    print('\n   AVISO: el calibrado es MÁS VIEJO que el espectro de C4 que calibra.')
    print('   D2 no se ha re-ejecutado desde la última corrida de C4, así que lo que')
    print('   hay en stages/ calibra un espectro que ya no está. Se arregla con:')
    print(f'      bash scripts/stage_x11_calibrate.sh --run-id {RUN_ID}')
    print('   (la §7 dirá DIFIERE por esto, no porque la copia se haya desviado)')


## 3 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal** del fuente, para que puedas editarla. Todo lo que viene después usa estos nombres locales, así que un cambio aquí se propaga al resultado — y la comparación de la §7 lo cuantifica.

- `VerificationError` — de `musepipe/reduction/verify.py`
- `circular_aperture_mask` — de `musepipe/reduction/verify.py`
- `extract_aperture_spectrum` — de `musepipe/reduction/verify.py`
- `TelluricError` — de `musepipe/reduction/telluric.py`
- `wavelength_axis_from_header` — de `musepipe/reduction/telluric.py`
- `CalibrationCorrections` — de `musepipe/stages/stage_x11_calibrate.py`
- `CalibratedProduct` — de `musepipe/stages/stage_x11_calibrate.py`
- `_finite_or_none` — de `musepipe/stages/stage_x11_calibrate.py`
- `_median_finite` — de `musepipe/stages/stage_x11_calibrate.py`
- `_scatter_frac` — de `musepipe/stages/stage_x11_calibrate.py`
- `_declared_fluxcal_from_m3` — de `musepipe/stages/stage_x11_calibrate.py`
- `_flux_scale_from_m3` — de `musepipe/stages/stage_x11_calibrate.py`
- `_wavelength_from_m1` — de `musepipe/stages/stage_x11_calibrate.py`
- `_psf_frac_from_qc` — de `musepipe/stages/stage_x11_calibrate.py`
- `_sky_frac_from_qc` — de `musepipe/stages/stage_x11_calibrate.py`
- `_telluric_fracs_from_qc` — de `musepipe/stages/stage_x11_calibrate.py`
- `calibration_corrections_from_qc` — de `musepipe/stages/stage_x11_calibrate.py`
- `audit_already_applied` — de `musepipe/stages/stage_x11_calibrate.py`
- `wavelength_offset_model` — de `musepipe/stages/stage_x11_calibrate.py`
- `apply_wavelength_correction` — de `musepipe/stages/stage_x11_calibrate.py`
- `conservative_stat_error` — de `musepipe/stages/stage_x11_calibrate.py`
- `_continuum_good_mask` — de `musepipe/stages/stage_x11_calibrate.py`
- `_window_mask` — de `musepipe/stages/stage_x11_calibrate.py`
- `_telluric_sys` — de `musepipe/stages/stage_x11_calibrate.py`
- `_error_budget_rows` — de `musepipe/stages/stage_x11_calibrate.py`
- `calibrate_spectrum_product` — de `musepipe/stages/stage_x11_calibrate.py`
- `calibrate_star_product` — de `musepipe/stages/stage_x11_calibrate.py`

> `stage_x11_paths` y `SpectrumProduct` se **importan**: son fontanería (dónde vive cada fichero, cómo se serializa), no la aritmética que se quiere tocar.


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from astropy.io import fits
from dataclasses import dataclass
from dataclasses import field
from musepipe.extraction.aperture import FLAG_BAD_WINDOW
from musepipe.extraction.aperture import FLAG_SKYLINE
from musepipe.extraction.product import SpectrumProduct
from musepipe.reduction.sky_zap import SKYLINE_WINDOWS
from musepipe.reduction.telluric import TELLURIC_BANDS
from musepipe.spectral import continuum_polyfit_loglambda
from musepipe.spectral import continuum_running_median
from musepipe.spectral import median_filter_1d
from musepipe.spectral import standard_line_free_mask
from pathlib import Path
import numpy as np

BAD_CONTINUUM_FLAGS = FLAG_BAD_WINDOW | FLAG_SKYLINE


class VerificationError(RuntimeError):
    """Raised when a requested verification cannot be completed."""


def circular_aperture_mask(shape: tuple[int, int], yx: tuple[float, float], radius: float) -> np.ndarray:
    y, x = np.indices(shape, dtype=np.float64)
    cy, cx = yx
    return (y - cy) ** 2 + (x - cx) ** 2 <= radius**2


def extract_aperture_spectrum(cube: np.ndarray, yx: tuple[float, float], radius: float) -> np.ndarray:
    mask = circular_aperture_mask(cube.shape[1:], yx, radius)
    if not mask.any():
        raise VerificationError("Aperture contains no pixels.")
    return np.nansum(cube[:, mask], axis=1)


class TelluricError(RuntimeError):
    """Raised when A3 must stop at a gate or checkpoint."""


def wavelength_axis_from_header(header: fits.Header, n_wave: int) -> np.ndarray:
    if all(key in header for key in ("CRVAL3", "CDELT3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CDELT3"])
    if all(key in header for key in ("CRVAL3", "CD3_3")):
        crpix = float(header.get("CRPIX3", 1.0))
        return float(header["CRVAL3"]) + (
            np.arange(int(n_wave), dtype=np.float64) + 1.0 - crpix
        ) * float(header["CD3_3"])
    raise TelluricError("Could not recover wavelength axis from DATA header.")


@dataclass(frozen=True)
class CalibrationCorrections:
    wavelength_status: str = "unavailable"
    wavelength_offset_A: float = 0.0
    wavelength_linear_a_A: float = np.nan
    wavelength_linear_b: float = np.nan
    wavelength_source: str = "stage00q_qc.m1_wavelength"
    wavelength_apply: bool = False
    frame_final: str = "unknown"
    flux_scale: float = 1.0
    flux_scale_err_frac: float = 0.0
    flux_source: str = "stage00q_qc.m3_flux"
    #: Calibracion absoluta DECLARADA y no plegada en `flux_err_total`: A4/M3 da
    #: `flux_factor` sin barra de error, asi que `flux_scale_err_frac` (el termino
    #: que si entra en el presupuesto) sale 0 y el desvio medido frente a Gaia se
    #: quedaba invisible. Se conserva como |1 - flux_factor| en su propia columna
    #: para poder citarlo sin mover el error de la ciencia congelada.
    flux_declared_err_frac: float = 0.0
    flux_declared_source: str = ""
    variability_caveat: bool = True
    psf_frac: float = 0.0
    psf_source: str = "stage_e01_qc.companion_ring_metric.residual_pct_median"
    sky_frac: float = 0.0
    sky_windows_A: tuple[tuple[float, float], ...] = tuple(SKYLINE_WINDOWS)
    sky_source: str = "stage00s_qc.systematic_frac"
    telluric_frac_by_band: dict[str, float] = field(default_factory=dict)
    telluric_bands_A: dict[str, tuple[float, float]] = field(default_factory=lambda: dict(TELLURIC_BANDS))
    telluric_source: str = "stage00t_qc.verification.v1_residual_pct_by_band"
    open_issues: tuple[str, ...] = ()


@dataclass(frozen=True)
class CalibratedProduct:
    method: str
    product: SpectrumProduct
    already_applied: dict
    continuum_summary: dict
    error_budget: list[dict]


def _finite_or_none(value):
    if value is None:
        return None
    val = float(value)
    if not np.isfinite(val):
        return None
    return val


def _median_finite(values, default=np.nan):
    arr = np.asarray(list(values), dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float(default)
    return float(np.nanmedian(arr))


def _scatter_frac(values):
    arr = np.asarray(list(values), dtype=np.float64)
    arr = arr[np.isfinite(arr) & (arr > 0)]
    if arr.size < 2:
        return 0.0
    med = float(np.nanmedian(arr))
    if med <= 0 or not np.isfinite(med):
        return 0.0
    return float(np.nanstd(arr) / med)


def _declared_fluxcal_from_m3(m3, err_frac):
    """El desvio de calibracion absoluta medido, para DECLARARLO sin aplicarlo.

    M3 compara el flujo sintetico de la primaria con el catalogo de Gaia y deja
    `flux_factor`, pero **sin barra de error**: `err_frac` sale 0 y el ~3% que
    M3 acaba de medir no aparece en ningun sitio del presupuesto. Se toma
    |1 - flux_factor| como la magnitud declarada de ese sistematico.

    No se pliega en `flux_err_total` a proposito: hacerlo cambiaria el error del
    compañero, que sostiene decisiones congeladas (E1/E3/G3). Nota: G3 ya asume
    su propio 10% de calibracion absoluta (`g3_sys_fluxcal_frac`), asi que este
    valor es una cota inferior de lo que el ajuste atmosferico ya se cree.
    """
    if err_frac > 0:
        # Si M3 llega a publicar una barra de error, esa manda y ya va aplicada.
        return 0.0, ""
    factor = m3.get("flux_factor") if m3 else None
    if factor is None:
        return 0.0, ""
    factor = float(factor)
    if not np.isfinite(factor) or factor <= 0:
        return 0.0, ""
    return (
        float(abs(1.0 - factor)),
        f"|1 - stage00q_qc.m3_flux.flux_factor| = |1 - {factor:.3f}| ({m3.get('band', '?')} band, "
        "vs Gaia DR3); declarado, NO sumado a flux_err_total",
    )


def _flux_scale_from_m3(m3):
    if not m3:
        return 1.0, 0.0, True, "stage00q_qc.m3_flux unavailable", ["A4/M3 flux QC unavailable; using scale=1."]
    issues = []
    if "scale_factor" in m3:
        scale = float(m3["scale_factor"])
        source = "stage00q_qc.m3_flux.scale_factor"
    elif "factor_median" in m3:
        scale = float(m3["factor_median"])
        source = "stage00q_qc.m3_flux.factor_median"
    elif "median_factor" in m3:
        scale = float(m3["median_factor"])
        source = "stage00q_qc.m3_flux.median_factor"
    elif isinstance(m3.get("factor_by_band"), dict):
        vals = [float(v) for v in m3["factor_by_band"].values()]
        median_factor = _median_finite(vals, default=1.0)
        scale = 1.0 / median_factor if np.isfinite(median_factor) and median_factor != 0 else 1.0
        source = "inverse(stage00q_qc.m3_flux.factor_by_band_median)"
    elif "flux_factor" in m3 and m3.get("flux_factor") is not None:
        # A4/M3 orchestrator (compute_m3_flux): synthetic/catalog flux factor.
        factor = float(m3["flux_factor"])
        if np.isfinite(factor) and 0.8 <= factor <= 1.25:
            # Consistent with 1 within M3's uncertainty (yellow/green): the cube
            # is absolute-flux-validated against Gaia; no correction applied and
            # NOT an open issue (a positive cross-check, recorded in source).
            scale = 1.0
            source = (f"stage00q_qc.m3_flux.flux_factor={factor:.3f} ({m3.get('band','?')} band) "
                      "validated vs Gaia DR3, consistent with 1 -> scale=1")
        else:
            scale = 1.0 / factor if factor != 0 else 1.0
            source = f"inverse(stage00q_qc.m3_flux.flux_factor={factor:.3f})"
            issues.append(f"A4/M3 flux factor {factor:.3f} off unity; applied scale={scale:.3f}.")
    else:
        scale = 1.0
        source = "stage00q_qc.m3_flux missing scale; scale=1"
        issues.append("A4/M3 flux scale unavailable; using scale=1.")
    err_frac = 0.0
    for key in ("scale_err_frac", "scale_error_frac", "flux_scale_err_frac"):
        if key in m3:
            err_frac = float(m3[key])
            break
    else:
        for key in ("scale_err_pct", "scale_error_pct", "factor_scatter_pct"):
            if key in m3:
                err_frac = float(m3[key]) / 100.0
                break
        else:
            if isinstance(m3.get("factor_by_band"), dict):
                err_frac = _scatter_frac([float(v) for v in m3["factor_by_band"].values()])
    declared_frac, declared_source = _declared_fluxcal_from_m3(m3, err_frac)
    return (float(scale), float(max(err_frac, 0.0)), bool(m3.get("variability_caveat", True)),
            source, issues, declared_frac, declared_source)


def _wavelength_from_m1(m1, *, allow_red=False):
    issues = []
    if not m1:
        issues.append("A4/M1 wavelength QC unavailable; no wavelength correction applied.")
        return "unavailable", 0.0, np.nan, np.nan, False, issues
    status = str(m1.get("status", "unavailable")).lower()
    if status == "red" and not allow_red:
        raise RuntimeError("A4/M1 wavelength status is red; D2 must stop before applying wavelength correction.")
    if status in {"unavailable", "unknown", "none"}:
        issues.append("A4/M1 wavelength correction unavailable; wavelength grid left in input frame.")
        return status, 0.0, np.nan, np.nan, False, issues
    offset = float(m1.get("offset_median_A", m1.get("dlambda_A", 0.0)))
    linear_a = float(m1.get("linear_a_A", np.nan))
    linear_b = float(m1.get("linear_b", m1.get("linear_term", np.nan)))
    return status, offset, linear_a, linear_b, True, issues


def _psf_frac_from_qc(qc):
    if not qc:
        return 0.0, ["C1 PSF QC unavailable; psf systematic set to zero."]
    metric = qc.get("companion_ring_metric", {})
    pct = metric.get("residual_pct_median", metric.get("median_pct", 0.0))
    if pct is None:
        return 0.0, ["C1 PSF QC lacks companion_ring_metric residual; psf systematic set to zero."]
    return max(float(pct), 0.0) / 100.0, []


def _sky_frac_from_qc(qc):
    if not qc:
        return 0.0, []
    for key in ("systematic_frac", "sky_systematic_frac", "residual_systematic_frac"):
        if key in qc:
            return max(float(qc[key]), 0.0), []
    decision = qc.get("decision", {})
    for key in ("systematic_frac", "sky_systematic_frac", "residual_systematic_frac"):
        if key in decision:
            return max(float(decision[key]), 0.0), []
    return 0.0, ["A2 sky QC has no explicit systematic fraction; sky term set to zero."]


def _telluric_fracs_from_qc(qc):
    if not qc:
        return {}, []
    fracs = {}
    explicit = qc.get("telluric_systematic_frac_by_band") or qc.get("systematic_frac_by_band")
    if isinstance(explicit, dict):
        for key, value in explicit.items():
            fracs[str(key)] = max(float(value), 0.0)
        return fracs, []
    residual = qc.get("verification", {}).get("v1_residual_pct_by_band", {})
    if isinstance(residual, dict):
        for key, value in residual.items():
            if value is not None:
                fracs[str(key)] = max(float(value), 0.0) / 100.0
    if fracs:
        return fracs, []
    return {}, ["A3 telluric QC has no residual/systematic by band; telluric term set to zero."]


def calibration_corrections_from_qc(qc00, qc_psf=None, qc_sky=None, qc_telluric=None, *, config=None):
    cfg = {} if config is None else dict(config)
    qc00 = {} if qc00 is None else qc00
    issues = []
    m1 = qc00.get("m1_wavelength", {})
    status, offset, linear_a, linear_b, apply_wl, wl_issues = _wavelength_from_m1(
        m1,
        allow_red=bool(cfg.get("x11_allow_red_wavelength", False)),
    )
    issues.extend(wl_issues)
    m3 = qc00.get("m3_flux", {})
    if str(m3.get("status", "unavailable")).lower() == "red" and not bool(cfg.get("x11_allow_red_flux", False)):
        raise RuntimeError("A4/M3 flux status is red; D2 must stop before applying flux scale.")
    (flux_scale, flux_err, variability, flux_source, flux_issues,
     flux_declared, flux_declared_source) = _flux_scale_from_m3(m3)
    issues.extend(flux_issues)
    psf_frac, psf_issues = _psf_frac_from_qc(qc_psf)
    issues.extend(psf_issues)
    sky_frac, sky_issues = _sky_frac_from_qc(qc_sky)
    issues.extend(sky_issues)
    telluric_fracs, telluric_issues = _telluric_fracs_from_qc(qc_telluric)
    issues.extend(telluric_issues)
    cube = qc00.get("cube", {})
    frame = str(cube.get("wavelength_frame", cfg.get("wavelength_frame", "unknown")))
    return CalibrationCorrections(
        wavelength_status=status,
        wavelength_offset_A=offset,
        wavelength_linear_a_A=linear_a,
        wavelength_linear_b=linear_b,
        wavelength_apply=apply_wl,
        frame_final=frame,
        flux_scale=flux_scale,
        flux_scale_err_frac=flux_err,
        flux_source=flux_source,
        flux_declared_err_frac=float(cfg.get("x11_fluxcal_declared_frac", flux_declared)),
        flux_declared_source=flux_declared_source,
        variability_caveat=variability,
        psf_frac=float(cfg.get("x11_psf_frac", psf_frac)),
        sky_frac=float(cfg.get("x11_sky_frac", sky_frac)),
        telluric_frac_by_band={**telluric_fracs, **cfg.get("x11_telluric_frac_by_band", {})},
        open_issues=tuple(issues),
    )


def audit_already_applied(product: SpectrumProduct, corrections: CalibrationCorrections | None = None) -> dict:
    product.validate()
    header = product.header
    missing = [key for key in ("WFRAME", "APCMODE", "ERRMODE") if key not in header]
    if missing:
        raise ValueError(f"SpectrumProduct header is ambiguous for D2; missing {missing}.")
    wframe = str(header["WFRAME"])
    if corrections is not None and corrections.frame_final not in {"unknown", "unavailable", "", None}:
        if wframe != str(corrections.frame_final):
            raise ValueError(f"Product WFRAME={wframe!r} differs from A4 frame {corrections.frame_final!r}.")
    errmode = str(header["ERRMODE"]).lower()
    stat_in_flux_err = errmode == "stat"
    if stat_in_flux_err:
        missing_stat = [key for key in ("STATFAC", "COVFAC") if key not in header]
        if missing_stat:
            raise ValueError(f"STAT error mode declared but header lacks {missing_stat}.")
    return {
        "wframe": wframe,
        "apcorr": True,
        "apcorr_mode": str(header["APCMODE"]),
        "stat_factors_in_flux_err": bool(stat_in_flux_err),
        "errmode": str(header["ERRMODE"]),
    }


def wavelength_offset_model(wave_A, corrections: CalibrationCorrections):
    wave = np.asarray(wave_A, dtype=np.float64)
    if not corrections.wavelength_apply:
        return np.zeros_like(wave)
    if np.isfinite(corrections.wavelength_linear_a_A) and np.isfinite(corrections.wavelength_linear_b):
        return corrections.wavelength_linear_a_A + corrections.wavelength_linear_b * wave
    return np.full_like(wave, float(corrections.wavelength_offset_A), dtype=np.float64)


def apply_wavelength_correction(wave_A, corrections: CalibrationCorrections):
    wave = np.asarray(wave_A, dtype=np.float64)
    return wave - wavelength_offset_model(wave, corrections)


def conservative_stat_error(flux_err, flux_err_emp, *, smooth_channels=21):
    stat = np.asarray(flux_err, dtype=np.float64)
    emp = np.asarray(flux_err_emp, dtype=np.float64)
    if stat.shape != emp.shape:
        raise ValueError("flux_err and flux_err_emp must have matching shapes.")
    combined = np.fmax(stat, emp)
    stat_only = np.isfinite(stat) & ~np.isfinite(emp)
    emp_only = np.isfinite(emp) & ~np.isfinite(stat)
    combined[stat_only] = stat[stat_only]
    combined[emp_only] = emp[emp_only]
    combined[(~np.isfinite(stat)) & (~np.isfinite(emp))] = np.nan
    return median_filter_1d(combined, smooth_channels)


def _continuum_good_mask(product, wave, flux):
    flags = np.asarray(product.flags, dtype=np.int32)
    base = np.isfinite(wave) & np.isfinite(flux) & ((flags & BAD_CONTINUUM_FLAGS) == 0)
    return standard_line_free_mask(wave, base)


def _window_mask(wave, windows):
    wave = np.asarray(wave, dtype=np.float64)
    mask = np.zeros(wave.size, dtype=bool)
    for lo, hi in windows or ():
        mask |= (wave >= float(lo)) & (wave <= float(hi))
    return mask


def _telluric_sys(wave, flux, corrections):
    out = np.zeros_like(np.asarray(flux, dtype=np.float64))
    for name, frac in corrections.telluric_frac_by_band.items():
        if name not in corrections.telluric_bands_A:
            continue
        lo, hi = corrections.telluric_bands_A[name]
        mask = (wave >= float(lo)) & (wave <= float(hi))
        out[mask] = np.maximum(out[mask], np.abs(flux[mask]) * float(frac))
    return out


def _error_budget_rows(flux_err_stat, sys_fluxcal, sys_psf, sys_sky, sys_telluric, sys_continuum,
                       corrections, sys_fluxcal_declared=None):
    rows = [
        {
            "term": "stat",
            "type": "per_channel",
            "median": _finite_or_none(np.nanmedian(flux_err_stat)),
            "source": "max(flux_err, flux_err_emp) smoothed",
        },
        {
            "term": "psf",
            "type": "localized",
            "median": _finite_or_none(np.nanmedian(sys_psf)),
            "value": float(corrections.psf_frac),
            "source": corrections.psf_source,
        },
        {
            "term": "fluxcal",
            "type": "global_pct",
            "value": float(corrections.flux_scale_err_frac),
            "median": _finite_or_none(np.nanmedian(sys_fluxcal)),
            "source": corrections.flux_source,
        },
        {
            "term": "sky",
            "type": "windows",
            "value": float(corrections.sky_frac),
            "median": _finite_or_none(np.nanmedian(sys_sky)),
            "source": corrections.sky_source,
        },
        {
            "term": "telluric",
            "type": "bands",
            "value": dict(corrections.telluric_frac_by_band),
            "median": _finite_or_none(np.nanmedian(sys_telluric)),
            "source": corrections.telluric_source,
        },
        {
            "term": "continuum",
            "type": "column_only",
            "median": _finite_or_none(np.nanmedian(sys_continuum)),
            "source": "D2 continuum runmed/poly difference",
        },
    ]
    if corrections.flux_declared_err_frac > 0:
        rows.append({
            "term": "fluxcal_declared",
            "type": "declared_not_applied",
            "value": float(corrections.flux_declared_err_frac),
            "median": (None if sys_fluxcal_declared is None
                       else _finite_or_none(np.nanmedian(sys_fluxcal_declared))),
            "source": corrections.flux_declared_source,
            "note": (
                "Columna `sys_fluxcal_declared`. NO entra en `flux_err_total`: plegarlo moveria el "
                "error del compañero, que sostiene decisiones congeladas (E1/E3/G3). G3 ya asume su "
                "propio 10% de calibracion absoluta (g3_sys_fluxcal_frac), mayor que este valor."
            ),
        })
    return rows


def calibrate_spectrum_product(
    product: SpectrumProduct,
    corrections: CalibrationCorrections,
    *,
    method: str,
    canonical: bool = False,
    continuum_window_A: float = 80.0,
    continuum_poly_deg: int = 5,
    error_smooth_channels: int = 21,
):
    already = audit_already_applied(product, corrections)
    wave = apply_wavelength_correction(product.wave_A, corrections)
    scale = float(corrections.flux_scale)
    flux = np.asarray(product.flux, dtype=np.float64) * scale
    flux_err_scaled = np.asarray(product.flux_err, dtype=np.float64) * abs(scale)
    flux_err_emp_scaled = np.asarray(product.flux_err_emp, dtype=np.float64) * abs(scale)
    flux_err_stat = conservative_stat_error(
        flux_err_scaled,
        flux_err_emp_scaled,
        smooth_channels=error_smooth_channels,
    )
    good_cont = _continuum_good_mask(product, wave, flux)
    min_pixels = max(5, min(15, int(np.count_nonzero(good_cont))))
    cont_runmed = continuum_running_median(
        wave,
        flux,
        good_cont,
        window_A=continuum_window_A,
        min_pixels=min_pixels,
    )
    cont_poly = continuum_polyfit_loglambda(
        wave,
        flux,
        good_cont,
        degree=continuum_poly_deg,
    )
    sys_continuum = np.abs(cont_runmed - cont_poly)
    sys_fluxcal = np.abs(flux) * float(corrections.flux_scale_err_frac)
    sys_psf = np.abs(flux) * float(corrections.psf_frac)
    sys_sky = np.zeros_like(flux)
    if corrections.sky_frac > 0:
        mask = _window_mask(wave, corrections.sky_windows_A)
        sys_sky[mask] = np.abs(flux[mask]) * float(corrections.sky_frac)
    sys_telluric = _telluric_sys(wave, flux, corrections)
    flux_err_total = np.sqrt(
        flux_err_stat**2 + sys_fluxcal**2 + sys_psf**2 + sys_sky**2 + sys_telluric**2
    )
    # DECLARADO y fuera de la suma: ver `_declared_fluxcal_from_m3`. Va despues
    # de `flux_err_total` justamente para que se vea que no entra en el.
    sys_fluxcal_declared = np.abs(flux) * float(corrections.flux_declared_err_frac)

    header = dict(product.header)
    header["SRCERRM"] = str(header.get("ERRMODE", "unknown"))
    header["ERRMODE"] = "total"
    header["CALSTAGE"] = "x11"
    header["CALMETH"] = str(method)
    header["CANON"] = bool(canonical)
    header["WLCORR"] = bool(corrections.wavelength_apply)
    header["DLAM_A"] = float(-corrections.wavelength_offset_A if corrections.wavelength_apply else 0.0)
    header["WLSRC"] = corrections.wavelength_source
    header["FLXSCL"] = float(scale)
    header["FLXSRC"] = corrections.flux_source
    header["SYSFLX"] = float(corrections.flux_scale_err_frac)
    header["SYSFLXD"] = float(corrections.flux_declared_err_frac)
    header["SYSFLXDN"] = "sys_fluxcal_declared NOT in flux_err_total"
    header["ERRTOT"] = "stat+sys"
    header["CONTRUN"] = float(continuum_window_A)
    header["CONTPOL"] = int(continuum_poly_deg)

    extra = dict(product.extra_columns or {})
    extra.update(
        {
            "flux_err_stat": flux_err_stat,
            "flux_err_total": flux_err_total,
            "cont_runmed": cont_runmed,
            "cont_poly": cont_poly,
            "sys_continuum": sys_continuum,
            "sys_fluxcal": sys_fluxcal,
            "sys_fluxcal_declared": sys_fluxcal_declared,
            "sys_psf": sys_psf,
            "sys_sky": sys_sky,
            "sys_telluric": sys_telluric,
        }
    )
    calibrated = SpectrumProduct(
        wave_A=wave,
        flux=flux,
        flux_err=flux_err_total,
        flux_err_emp=flux_err_emp_scaled,
        apcorr=np.asarray(product.apcorr, dtype=np.float64),
        npix_eff=np.asarray(product.npix_eff, dtype=np.float64),
        flags=np.asarray(product.flags, dtype=np.int32),
        header=header,
        covariance=None if product.covariance is None else np.asarray(product.covariance, dtype=np.float64) * scale**2,
        extra_columns=extra,
    )
    calibrated.validate()
    halpha = (wave >= 6540.0) & (wave <= 6590.0)
    ratio = np.nan
    if np.any(halpha):
        denom = np.nanmedian(flux_err_stat[halpha])
        numer = np.nanmedian(sys_continuum[halpha])
        if np.isfinite(denom) and denom > 0:
            ratio = float(numer / denom)
    stable = np.nan
    with np.errstate(invalid="ignore"):
        stable = float(np.nanmean((sys_continuum <= flux_err_stat)[good_cont])) if np.any(good_cont) else np.nan
    continuum_summary = {
        "runmed_window_A": float(continuum_window_A),
        "poly_deg": int(continuum_poly_deg),
        "sys_at_halpha_vs_staterr": _finite_or_none(ratio),
        "fraction_good_channels_sys_lt_staterr": _finite_or_none(stable),
    }
    budget = _error_budget_rows(
        flux_err_stat,
        sys_fluxcal,
        sys_psf,
        sys_sky,
        sys_telluric,
        sys_continuum,
        corrections,
        sys_fluxcal_declared=sys_fluxcal_declared,
    )
    return CalibratedProduct(
        method=method,
        product=calibrated,
        already_applied=already,
        continuum_summary=continuum_summary,
        error_budget=budget,
    )


def calibrate_star_product(cfg, paths, corrections):
    """Calibra el espectro de la PRIMARIA con la misma cadena que el compañero.

    Devuelve `(CalibratedProduct, resumen_qc)` o `(None, resumen_qc)` si C4 no
    dejo el producto (p.ej. una cadena que no corrio psffit).

    Los dos terminos de error quedan en columnas SEPARADAS, no fundidos:

    * `flux_err_emp` — empirico de anillo. Es la dispersion del coeficiente de
      la primaria entre los N ajustes psffit de control, colocados en un anillo
      a la separacion del compañero alrededor de la estrella
      (`extraction.psffit.control_psffit_spectra`). Mide cuanto se mueve el
      flujo de la primaria segun donde se ponga la segunda componente: es un
      sistematico del ajuste medido sobre el dato, no ruido de fotones.
    * `flux_err_stat` + `sys_fluxcal` / `sys_psf` / `sys_sky` / `sys_telluric` /
      `sys_continuum` — el presupuesto de sistematicos de D2, identico al del
      compañero.
    * `flux_err_total` — la suma en cuadratura de ambos bloques.

    La primaria tiene S/N enorme, asi que su error NO esta dominado por el
    termino estadistico sino por el presupuesto (calibracion absoluta de flujo
    de A4/M3, PSF, telurico). Por eso importa poder mirarlos por separado.
    """
    path = Path(cfg.get("x11_spec_psffit_star", paths["spec_psffit_star"]))
    if not path.exists():
        return None, {"available": False, "reason": f"C4 no dejo {path.name} en este run"}
    product = SpectrumProduct.read(path)
    calibrated = calibrate_spectrum_product(
        product,
        corrections,
        method="psffit",
        canonical=False,
        continuum_window_A=float(cfg.get("x11_continuum_window_A", 80.0)),
        continuum_poly_deg=int(cfg.get("x11_continuum_poly_deg", 5)),
        error_smooth_channels=int(cfg.get("x11_error_smooth_channels", 21)),
    )
    header = calibrated.product.header
    header["SOURCE"] = "primary"
    header["EMPSRC"] = "psffit control ring at the companion separation (C4)"
    header["ERRSEP"] = "flux_err_emp (ring) and sys_* (budget) kept separate"

    flux = np.asarray(calibrated.product.flux, dtype=np.float64)
    extra = calibrated.product.extra_columns or {}
    summary = {
        "available": True,
        "input": str(path),
        "output": str(paths["spec_calibrated_psffit_star"]),
        "n_channels": int(flux.size),
        "median_snr_total": _finite_or_none(
            _median_finite(np.abs(flux) / np.asarray(extra["flux_err_total"], dtype=np.float64))
        ),
        "median_snr_emp_only": _finite_or_none(
            _median_finite(np.abs(flux) / np.asarray(calibrated.product.flux_err_emp, dtype=np.float64))
        ),
        "error_terms": {
            "empirical_ring": "flux_err_emp",
            "budget": ["flux_err_stat", "sys_fluxcal", "sys_psf", "sys_sky",
                       "sys_telluric", "sys_continuum"],
            "combined": "flux_err_total",
            "declared_not_combined": ["sys_fluxcal_declared"],
        },
        "median_error_fraction": {
            name: _finite_or_none(
                _median_finite(np.asarray(extra[name], dtype=np.float64) / np.abs(flux))
            )
            for name in ("flux_err_stat", "sys_fluxcal", "sys_fluxcal_declared", "sys_psf",
                         "sys_sky", "sys_telluric", "sys_continuum")
        },
        "median_empirical_fraction": _finite_or_none(
            _median_finite(np.asarray(calibrated.product.flux_err_emp, dtype=np.float64) / np.abs(flux))
        ),
        "note": (
            "La primaria es la fuente brillante: su error lo domina el presupuesto de "
            "sistematicos, no el termino estadistico. El empirico de anillo se conserva "
            "aparte porque mide otra cosa (estabilidad del ajuste), no ruido de fotones."
        ),
    }
    return calibrated, summary


## 4 · Chequeo de deriva

Compara el fuente copiado arriba con el que **hoy** tiene `musepipe`. Si alguien cambió la cadena, esta celda lo dice nombrando la función: es lo que evita que este notebook siga dando resultados «de la cadena» cuando ya no lo son.


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/reduction/verify.py:VerificationError": "d9acb4457343",
    "musepipe/reduction/verify.py:circular_aperture_mask": "b08d990cd3a2",
    "musepipe/reduction/verify.py:extract_aperture_spectrum": "e46a9616dd50",
    "musepipe/reduction/telluric.py:TelluricError": "5347f3394587",
    "musepipe/reduction/telluric.py:wavelength_axis_from_header": "f463214b3a31",
    "musepipe/stages/stage_x11_calibrate.py:CalibrationCorrections": "c7b07e9e7975",
    "musepipe/stages/stage_x11_calibrate.py:CalibratedProduct": "41a81fe64fe3",
    "musepipe/stages/stage_x11_calibrate.py:_finite_or_none": "74207600dd12",
    "musepipe/stages/stage_x11_calibrate.py:_median_finite": "0fc574dcde28",
    "musepipe/stages/stage_x11_calibrate.py:_scatter_frac": "c414840be08a",
    "musepipe/stages/stage_x11_calibrate.py:_declared_fluxcal_from_m3": "d7ae82669319",
    "musepipe/stages/stage_x11_calibrate.py:_flux_scale_from_m3": "667e4848b271",
    "musepipe/stages/stage_x11_calibrate.py:_wavelength_from_m1": "0f5154b0e6d8",
    "musepipe/stages/stage_x11_calibrate.py:_psf_frac_from_qc": "c13c450ebb39",
    "musepipe/stages/stage_x11_calibrate.py:_sky_frac_from_qc": "1720fba866b0",
    "musepipe/stages/stage_x11_calibrate.py:_telluric_fracs_from_qc": "b5266b0904ba",
    "musepipe/stages/stage_x11_calibrate.py:calibration_corrections_from_qc": "4c1eff11eb9f",
    "musepipe/stages/stage_x11_calibrate.py:audit_already_applied": "1df4a17350e9",
    "musepipe/stages/stage_x11_calibrate.py:wavelength_offset_model": "f1518e5862ac",
    "musepipe/stages/stage_x11_calibrate.py:apply_wavelength_correction": "6d4abab991af",
    "musepipe/stages/stage_x11_calibrate.py:conservative_stat_error": "613d560a5469",
    "musepipe/stages/stage_x11_calibrate.py:_continuum_good_mask": "8802e2be676d",
    "musepipe/stages/stage_x11_calibrate.py:_window_mask": "c0533a71c801",
    "musepipe/stages/stage_x11_calibrate.py:_telluric_sys": "90fb12915a03",
    "musepipe/stages/stage_x11_calibrate.py:_error_budget_rows": "fb08fc6ed5ac",
    "musepipe/stages/stage_x11_calibrate.py:calibrate_spectrum_product": "a17dab79668e",
    "musepipe/stages/stage_x11_calibrate.py:calibrate_star_product": "24008b0102af",
    "musepipe/stages/stage_x11_calibrate.py:BAD_CONTINUUM_FLAGS": "4b0e9c5eb5e9"
}

def _pieza(cuerpo, name):
    """El nodo que define `name`: def/class, o la asignación de una constante.

    Las constantes también se vigilan: viajan copiadas igual que las
    funciones, y hasta ahora nadie comprobaba que siguieran siendo las de
    `musepipe` — añadir una banda a un diccionario dejaba esta copia atrás
    sin que nada lo dijera.
    """
    for n in cuerpo:
        if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name:
            inicio = min([n.lineno] + [d.lineno for d in n.decorator_list])
            return inicio, n.end_lineno
        if isinstance(n, _ast.Assign) and any(
                isinstance(t, _ast.Name) and t.id == name for t in n.targets):
            return n.lineno, n.end_lineno
        if (isinstance(n, _ast.AnnAssign) and isinstance(n.target, _ast.Name)
                and n.target.id == name):
            return n.lineno, n.end_lineno
    return None

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        sitio = _pieza(_ast.parse(text).body, name)
        if sitio is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        inicio, fin = sitio
        src = ''.join(lines[inicio - 1:fin]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} D2')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 5 · Qué es «el espectro de la primaria» antes de calibrarlo

No es una apertura. C4 ajusta **dos PSF a la vez** en cada canal —la primaria y el compañero, con la forma cromática que midió C1— y el espectro de la estrella es la serie de **coeficientes de la componente primaria**, canal a canal. Por eso viene ya con corrección de apertura aplicada (`APCMODE` en la cabecera) y por eso su error empírico, `flux_err_emp`, no es ruido de fotones sino **cuánto se mueve ese coeficiente** según dónde se ponga la segunda componente: C4 repite el ajuste con el compañero desplazado a un anillo de posiciones de control a la misma separación, y la dispersión entre esos ajustes es la columna.

Lo que hay que mirar aquí: que el espectro sea **suave y positivo** (es una estrella brillante: cualquier estructura estrecha que no sea una línea conocida es del método), y el **contraste enorme** entre los dos errores — el empírico es minúsculo comparado con lo que después le añadirá el presupuesto de sistemáticos.


In [ ]:
if not ENTRADA_C4.exists():
    raise SystemExit(f'C4 no dejó {ENTRADA_C4.name} en este run: ejecuta C4 antes.')

ESTRELLA = SpectrumProduct.read(ENTRADA_C4)
W0 = np.asarray(ESTRELLA.wave_A, dtype=np.float64)
F0 = np.asarray(ESTRELLA.flux, dtype=np.float64)
E0 = np.asarray(ESTRELLA.flux_err, dtype=np.float64)
EMP0 = np.asarray(ESTRELLA.flux_err_emp, dtype=np.float64)
BUNIT = ESTRELLA.header.get('BUNIT') or 'sin unidad declarada'

print('columnas   :', ['flux', 'flux_err', 'flux_err_emp', 'apcorr', 'npix_eff', 'flags']
      + sorted(ESTRELLA.extra_columns or {}))
print('canales    :', F0.size, ' finitos:', int(np.isfinite(F0).sum()))
print('BUNIT      :', BUNIT)
for _k in ('WFRAME', 'APCMODE', 'ERRMODE', 'STATFAC', 'COVFAC', 'FORMATV'):
    if _k in ESTRELLA.header:
        print(f'{_k:11s}:', ESTRELLA.header[_k])
with np.errstate(invalid='ignore', divide='ignore'):
    print(f'S/N (empírico) mediano: {np.nanmedian(np.abs(F0) / EMP0):.0f}')

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11.5, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2.2, 1]})
for _n, (_lo, _hi) in TELLURIC_BANDS.items():
    a1.axvspan(_lo, _hi, color='tab:orange', alpha=0.12, zorder=0)
a1.axvline(6562.8, color='tab:red', lw=0.8, ls=':', zorder=1)
a1.plot(W0, F0, lw=0.6, color='tab:blue')
a1.set_ylabel(f'flujo [{BUNIT}]', fontsize=8)
a1.set_title('la primaria tal como sale de C4 (naranja: bandas telúricas; '
             'punteada: Hα)', fontsize=9)
with np.errstate(invalid='ignore', divide='ignore'):
    a2.plot(W0, 100 * EMP0 / np.abs(F0), lw=0.6, color='tab:purple',
            label='empírico de anillo')
    a2.plot(W0, 100 * E0 / np.abs(F0), lw=0.6, color='0.6', label='flux_err de C4')
a2.set_yscale('log'); a2.legend(fontsize=7)
a2.set_ylabel('error [% del flujo]', fontsize=8); a2.set_xlabel('λ [Å]', fontsize=8)
fig.tight_layout(); plt.show()


## 6 · Las correcciones, una por una

D2 no re-extrae nada: toma el espectro de C4 y le aplica **lo que midieron otras etapas**. Cada término tiene una procedencia declarada, y la celda la imprime al lado del número — es la parte del producto que se puede auditar sin volver al dato.

- **λ** — el desplazamiento que midió A4/M1 sobre líneas de cielo. Se aplica al eje, no al flujo.
- **escala de flujo** — A4/M3 compara el flujo sintético de esta misma estrella con el catálogo de Gaia. Si el factor sale consistente con 1 dentro de su incertidumbre, **no se aplica** y queda como control positivo; el desvío medido se declara aparte (`sys_fluxcal_declared`) precisamente para no mover resultados congelados con él.
- **PSF, cielo, telúrico** — no son correcciones sino **sistemáticos**: fracciones de flujo que se suman en cuadratura al error, cada una donde toca (el telúrico solo dentro de sus bandas, el cielo solo en las ventanas de línea de cielo).

> Los `open_issues` de abajo no son excepciones: son las entradas que **faltaban** y cuyo término se ha puesto a cero. Un cero por «no medido» y un cero por «medido y vale cero» no son lo mismo, y esa lista es la única que los distingue.


In [ ]:
CORR = calibration_corrections_from_qc(
    QC00, qc_psf=QC_PSF, qc_sky=QC_SKY, qc_telluric=QC_TEL, config=X11)

print(f'{"término":16s}{"valor":>12s}   procedencia')
_dlam = -CORR.wavelength_offset_A if CORR.wavelength_apply else 0.0
print(f'{"λ (Δλ aplicado)":16s}{_dlam:>11.4f}Å   {CORR.wavelength_source}'
      f'  [{CORR.wavelength_status}]')
print(f'{"escala flujo":16s}{CORR.flux_scale:>12.4f}   {CORR.flux_source}')
print(f'{"σ flujo (aplic.)":16s}{100 * CORR.flux_scale_err_frac:>11.2f}%   '
      f'entra en flux_err_total')
print(f'{"σ flujo (decl.)":16s}{100 * CORR.flux_declared_err_frac:>11.2f}%   '
      f'{CORR.flux_declared_source or "—"}')
print(f'{"σ PSF":16s}{100 * CORR.psf_frac:>11.2f}%   {CORR.psf_source}')
print(f'{"σ cielo":16s}{100 * CORR.sky_frac:>11.2f}%   {CORR.sky_source}')
print(f'{"σ telúrico":16s}{str(dict(CORR.telluric_frac_by_band)) or "{}":>12s}   '
      f'{CORR.telluric_source}')
print(f'{"marco λ final":16s}{CORR.frame_final:>12s}')

if CORR.open_issues:
    print('\nentradas que faltaban (su término va a CERO por ausencia, no por medida):')
    for _i in CORR.open_issues:
        print('  ·', _i)
else:
    print('\nsin entradas ausentes: todos los términos están medidos.')

# El desplazamiento en λ, dibujado: es lineal en λ o constante, segun lo que
# midiera M1, y conviene ver cual de los dos porque cambia el borde azul.
_off = wavelength_offset_model(W0, CORR)
if np.any(_off != 0):
    fig, ax = plt.subplots(figsize=(9, 2.4))
    ax.plot(W0, _off, lw=1.2, color='tab:green')
    ax.axhline(0, color='0.7', lw=0.6)
    ax.set_xlabel('λ [Å]', fontsize=8); ax.set_ylabel('offset [Å]', fontsize=8)
    ax.set_title('lo que se le resta al eje de λ (modelo de A4/M1)', fontsize=9)
    ax.tick_params(labelsize=7); fig.tight_layout(); plt.show()
else:
    print('\nsin corrección de λ: el eje queda en el marco de entrada.')


## 7 · La calibración rehecha aquí, y la comparación con la cadena

Contra `spec_calibrated_psffit_star.fits`, el producto que escribió la etapa. **Con las perillas por defecto debe salir idéntico** (a precisión de coma flotante): si no lo es, o la copia se desvió, o alguna entrada no es la que usó la cadena, o el producto en disco es más viejo que su entrada (lo que avisó la §2). En cuanto cambias una perilla, esta celda mide exactamente qué se movió.


In [ ]:
MIO, RESUMEN = calibrate_star_product(X11, PATHS, CORR)
PRIM = MIO.product
WAVE = np.asarray(PRIM.wave_A, dtype=np.float64)
FLUJO = np.asarray(PRIM.flux, dtype=np.float64)
EXTRA = dict(PRIM.extra_columns or {})

if not SALIDA_D2.exists():
    print(f'la cadena no ha dejado {SALIDA_D2.name}: no hay contra qué comparar.')
    print('ejecuta D2 con:  bash scripts/stage_x11_calibrate.sh --run-id', RUN_ID)
else:
    CADENA = SpectrumProduct.read(SALIDA_D2)

    def _compara(nombre, mio, suyo, rtol=1e-9):
        mio = np.asarray(mio, dtype=np.float64); suyo = np.asarray(suyo, dtype=np.float64)
        if mio.shape != suyo.shape:
            print(f'  {nombre:16s} FORMA distinta {mio.shape} vs {suyo.shape}')
            return False
        finito = np.isfinite(mio) & np.isfinite(suyo)
        if not finito.any():
            print(f'  {nombre:16s} sin canales finitos en común'); return False
        dif = np.abs(mio - suyo)[finito]
        escala = np.maximum(np.abs(suyo)[finito], 1e-30)
        iguales = np.isclose(mio[finito], suyo[finito], rtol=rtol, atol=0.0)
        print(f'  {nombre:16s} idénticos {100 * iguales.mean():6.2f}% de {finito.sum()}'
              f' canales | máx |Δ| = {dif.max():.3e} ({100 * (dif / escala).max():.2e}%)')
        return bool(iguales.all())

    print('mi resultado vs la cadena:')
    ok = _compara('λ', WAVE, CADENA.wave_A)
    ok &= _compara('flujo', FLUJO, CADENA.flux)
    ok &= _compara('error total', PRIM.flux_err, CADENA.flux_err)
    ok &= _compara('error empírico', PRIM.flux_err_emp, CADENA.flux_err_emp)
    ok &= _compara('apcorr', PRIM.apcorr, CADENA.apcorr)
    _suyas = dict(CADENA.extra_columns or {})
    for _c in sorted(set(EXTRA) & set(_suyas)):
        ok &= _compara(_c, EXTRA[_c], _suyas[_c])
    print()
    print('IDÉNTICO: la copia reproduce la cadena.' if ok else
          'DIFIERE — si has tocado una perilla, es lo esperado; si no, mira el aviso '
          'de la §2 (producto más viejo que su entrada) y el chequeo de deriva.')
    if not ok and not D2_AL_DIA:
        print('   → en este run el calibrado en disco es más viejo que su entrada de C4.')

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11.5, 5.2), sharex=True,
                                 gridspec_kw={'height_ratios': [2, 1]})
    a1.plot(CADENA.wave_A, median_filter_1d(np.asarray(CADENA.flux, float), 41),
            lw=1.6, color='0.6', label='cadena')
    a1.plot(WAVE, median_filter_1d(FLUJO, 41), lw=1.0, color='tab:blue', ls='--',
            label='este notebook')
    a1.legend(fontsize=8); a1.set_ylabel(f'flujo (mediana 41 ch)', fontsize=8)
    _n = min(FLUJO.size, np.asarray(CADENA.flux).size)
    a2.plot(WAVE[:_n], FLUJO[:_n] - np.asarray(CADENA.flux, float)[:_n],
            lw=0.7, color='tab:purple')
    a2.axhline(0, color='0.7', lw=0.6)
    a2.set_ylabel('este − cadena', fontsize=8); a2.set_xlabel('λ [Å]', fontsize=8)
    a1.set_title('la primaria calibrada: comparación con el producto de la cadena',
                 fontsize=9)
    fig.tight_layout(); plt.show()


## 8 · De qué está hecha la barra de error de la primaria

La primaria tiene S/N enorme, así que su error **no** lo domina el término estadístico: lo dominan los sistemáticos. Esto importa para leer cualquier cosa que se mida sobre ella (una línea, un color, un cociente con el compañero): un rasgo del 1 % es significativo frente al ruido y **no** frente al presupuesto.

Los dos bloques se mantienen separados a propósito:

- `flux_err_emp` — el **empírico de anillo** de C4: cuánto se mueve el coeficiente de la primaria según dónde se ponga la segunda componente. Mide la **estabilidad del ajuste**, no ruido de fotones.
- `flux_err_stat` + los `sys_*` — el presupuesto de D2, idéntico al del compañero.
- `sys_fluxcal_declared` va **fuera** de `flux_err_total`: plegarlo movería el error del compañero, que sostiene decisiones congeladas (E1/E3/G3).


In [ ]:
print(f'{"término":24s}{"mediana [% del flujo]":>22s}')
_frac = RESUMEN.get('median_error_fraction', {})
for _t in ('flux_err_stat', 'sys_psf', 'sys_continuum', 'sys_fluxcal',
           'sys_sky', 'sys_telluric', 'sys_fluxcal_declared'):
    _v = _frac.get(_t)
    _nota = '  (DECLARADO, no sumado)' if _t.endswith('declared') else ''
    print(f'{_t:24s}{100 * _v if _v is not None else float("nan"):>21.3f}%{_nota}')
print(f'{"flux_err_emp (anillo)":24s}'
      f'{100 * (RESUMEN.get("median_empirical_fraction") or float("nan")):>21.3f}%')
print(f'\nS/N mediano con el error TOTAL : {RESUMEN.get("median_snr_total"):.1f}')
print(f'S/N mediano solo con el empírico: {RESUMEN.get("median_snr_emp_only"):.1f}')
print('\n(la distancia entre esos dos números ES el presupuesto de sistemáticos)')

_terminos = [('flux_err_stat', 'estadístico', 'tab:blue'),
             ('sys_psf', 'PSF', 'tab:red'),
             ('sys_continuum', 'continuo', 'tab:green'),
             ('sys_fluxcal', 'calib. flujo', 'tab:orange'),
             ('sys_sky', 'cielo', 'tab:purple'),
             ('sys_telluric', 'telúrico', 'tab:brown')]
fig, ax = plt.subplots(figsize=(11.5, 3.6))
with np.errstate(invalid='ignore', divide='ignore'):
    for _c, _lab, _col in _terminos:
        if _c in EXTRA and np.any(np.asarray(EXTRA[_c]) != 0):
            ax.plot(WAVE, 100 * np.asarray(EXTRA[_c], float) / np.abs(FLUJO),
                    lw=0.7, color=_col, label=_lab)
    ax.plot(WAVE, 100 * np.asarray(PRIM.flux_err, float) / np.abs(FLUJO),
            lw=1.3, color='k', label='TOTAL')
    ax.plot(WAVE, 100 * np.asarray(PRIM.flux_err_emp, float) / np.abs(FLUJO),
            lw=0.9, color='0.5', ls='--', label='empírico (anillo)')
ax.set_yscale('log'); ax.legend(fontsize=7, ncol=4)
ax.set_xlabel('λ [Å]', fontsize=8); ax.set_ylabel('% del flujo', fontsize=8)
ax.set_title('el presupuesto de error de la primaria, término a término', fontsize=9)
fig.tight_layout(); plt.show()


## 9 · El espectro de **cada observación**

Todo lo anterior sale de **un** cubo: el combinado. Esta sección abre esa caja y mide la primaria en **cada exposición por separado**, que es lo único que permite responder a tres preguntas que el combinado ya no deja hacer:

1. **¿Son comparables las observaciones?** La AO de NFM no rinde igual toda la noche: si el núcleo de la PSF cambia, el flujo dentro de una apertura fija cambia con él **sin que la estrella haya hecho nada**.
2. **¿Cambió la estrella?** La primaria es la fuente de acreción brillante del sistema. Variabilidad en Hα entre exposiciones es física, no instrumento — y es un aviso para cualquier medida sobre el combinado, que promedia lo que haya pasado.
3. **¿Qué peso tiene cada observación en el combinado?** Si una exposición aporta el doble de señal que otra, el «espectro medio» no es el de la mitad de la noche.

**Cómo se mide, y por qué así.** La posición de la estrella **no se hereda** del combinado: cada exposición tiene su propio marco (el dither y el alineado son posteriores), así que copiar el `primary_yx` del combinado mediría un trozo de cielo distinto en cada una. Se localiza en cada cubo por su propio máximo y se centra por centroide, exactamente como hace `scripts/molecfit_granularity.py`. La extracción es la **misma función** que usa A3 (`extract_aperture_spectrum`, copiada arriba y vigilada por el chequeo de deriva).

> **Un aviso de procedencia que hay que leer antes que los números.** Los cubos por exposición que hay en disco **no tienen por qué ser todos los que entraron en el combinado**, ni venir de la misma reducción. La celda siguiente lo comprueba y lo dice: cuenta las exposiciones que el plan de reducción declara y las compara con los cubos disponibles. Si no coinciden, lo que sigue describe **esas** observaciones, no el combinado.


In [ ]:
def _cubos_por_observacion():
    """(etiqueta, ruta) de cada exposicion, DECLARADAS en el config del run.

    Nunca por convencion de nombres sobre un disco: `perexp_cubes` (lista) o
    `perexp_dir` (raiz) viven en el config porque son propias del objeto.
    """
    lista = CFG.get('perexp_cubes') or nb._sibling_setting(RUN_ID, 'perexp_cubes')
    if not lista:
        raiz = CFG.get('perexp_dir') or nb._sibling_setting(RUN_ID, 'perexp_dir')
        if raiz:
            lista = sorted(str(p) for p in Path(raiz).glob('*/DATACUBE_FINAL.fits'))
    presentes = [Path(_p) for _p in (lista or []) if Path(_p).exists()]
    return list(zip(_etiquetas(presentes), presentes))

def _etiquetas(rutas):
    """Un nombre corto y UNICO por cubo, sacado de su directorio.

    Los arboles de reduccion no comparten convencion: unos usan `exp1..expN`
    y otros el identificador de la exposicion
    (`2022-08-29_MUSE.2022-08-29T23:22:03.446`). Tomar `p.stem` cuando no
    empieza por `exp` daba `DATACUBE_FINAL` para TODOS y las 29
    observaciones colapsaban en una.
    """
    crudas = [p.parent.name for p in rutas]
    if all(c.lower().startswith('exp') for c in crudas) and len(set(crudas)) == len(crudas):
        return crudas
    # `<noche>_MUSE.<fecha>T<hora>` -> `MM-DD#NN`, con NN correlativo
    # dentro de su noche: corto, ordenable y agrupa por OB a simple vista.
    import re as _re
    salida, cuenta = [], {}
    for c in crudas:
        m = _re.match(r'(\d{4})-(\d{2})-(\d{2})', c)
        noche = f'{m.group(2)}-{m.group(3)}' if m else c[:8]
        cuenta[noche] = cuenta.get(noche, 0) + 1
        salida.append(f'{noche}#{cuenta[noche]:02d}')
    return salida if len(set(salida)) == len(salida) else crudas

OBSERVACIONES = _cubos_por_observacion()
# El cubo combinado: el que A3 declara haber medido, que es el de la cadena.
_decl = ((QC_TEL or {}).get('input') or {}).get('cube') or (CFG.get('cube_files') or [None])[0]
COMBINADO = Path(_decl) if _decl else None
if COMBINADO is not None and not COMBINADO.is_absolute():
    COMBINADO = ROOT / COMBINADO

print(f'cubos por observación disponibles: {len(OBSERVACIONES)}')
for _e, _p in OBSERVACIONES:
    print(f'   {_e:8s} {_p}')
print('combinado:', COMBINADO if (COMBINADO and COMBINADO.exists()) else 'no disponible')

# ¿Cuantas exposiciones hay DETRAS del combinado? El plan de la reduccion lo
# sabe; si no esta declarado no se inventa, se dice que no se sabe.
_plan = CFG.get('perexp_execution') or nb._sibling_setting(RUN_ID, 'perexp_execution')
N_EN_COMBINADO, NOCHES = None, set()
if _plan and Path(_plan).exists():
    _e = json.loads(Path(_plan).read_text(encoding='utf-8'))
    _exps = _e.get('exposures') or {}
    _claves = list(_exps) if isinstance(_exps, dict) else [str(x) for x in _exps]
    N_EN_COMBINADO = len(_claves)
    NOCHES = {k.split('_')[0] for k in _claves if '_' in k}
    print(f'\nel plan de reducción declara {N_EN_COMBINADO} exposiciones'
          f"{' en ' + str(len(NOCHES)) + ' noches: ' + ', '.join(sorted(NOCHES)) if NOCHES else ''}")
    if OBSERVACIONES and N_EN_COMBINADO != len(OBSERVACIONES):
        print(f'   AVISO: hay {len(OBSERVACIONES)} cubos por exposición para '
              f'{N_EN_COMBINADO} exposiciones combinadas.')
        print('   Lo que mide esta sección describe ESAS observaciones, no el combinado.')
else:
    print('\nno hay plan de reducción declarado (`perexp_execution`): no se puede saber')
    print('cuántas exposiciones hay detrás del combinado.')

if not OBSERVACIONES:
    print('\nEste objeto no declara cubos por exposición. Para tenerlos, añade al')
    print('config del run  "perexp_cubes": [...]  o  "perexp_dir": "..."  y')
    print('genera los cubos con  python scripts/regen_perexp_cubes.py --step all')


### 9.1 · Qué observación es cada una

Antes de mirar espectros, las condiciones. `EXPTIME`, la **masa de aire** (la que manda en el telúrico) y el **seeing del DIMM** vienen de la cabecera; la **FWHM de la imagen blanca** se mide aquí, sobre el propio cubo, porque es la que de verdad gobierna cuánta luz cae dentro de la apertura — y el DIMM y la PSF entregada por la AO pueden no parecerse en nada.

La lectura de cada cubo es de unos pocos segundos: se lee **solo una ventana** alrededor de la estrella, no el cubo entero.


In [ ]:
def _cabecera_de(cubo):
    """Lo que la exposicion sabe de si misma. `None` donde no lo declara."""
    with fits.open(cubo, memmap=True) as h:
        pri = h[0].header
        hdu = h['DATA'] if 'DATA' in h else h[1]
        dat = hdu.header
        forma = tuple(int(v) for v in hdu.shape)
    _x = [pri.get('HIERARCH ESO TEL AIRM START'), pri.get('HIERARCH ESO TEL AIRM END')]
    _x = [float(v) for v in _x if v is not None]
    return {'date_obs': pri.get('DATE-OBS') or dat.get('DATE-OBS'),
            'exptime': pri.get('EXPTIME') or dat.get('EXPTIME'),
            'airmass': float(np.mean(_x)) if _x else None,
            'seeing': pri.get('HIERARCH ESO TEL AMBI FWHM START'),
            # El OB es la unidad de observación que fija
            # `docs/2026-08-06_indicaciones_reduccion.md`: dos exposiciones de
            # OB distintos son dos noches distintas, y promediarlas es una
            # decisión, no un detalle de bookkeeping.
            'ob_id': pri.get('HIERARCH ESO OBS ID'),
            'ob_name': pri.get('HIERARCH ESO OBS NAME'),
            'bunit': dat.get('BUNIT'), 'forma': forma}

def _primaria_de(cubo, radios=RADIOS_PX, half=VENTANA_PX, paso=PASO_BLANCO):
    """(wave, {radio: espectro}, (y,x), fwhm_px) de la primaria EN ESE cubo.

    La posicion se mide en el propio cubo. Se lee una ventana de
    (2*half+1)^2 pixeles en todos los canales, no el cubo entero.
    """
    with fits.open(cubo, memmap=True) as h:
        hdu = h['DATA'] if 'DATA' in h else h[1]
        blanco = np.nanmedian(np.asarray(hdu.data[::paso], dtype=np.float64), axis=0)
        y0, x0 = (int(v) for v in np.unravel_index(np.nanargmax(blanco), blanco.shape))
        sy = slice(max(0, y0 - 3), y0 + 4); sx = slice(max(0, x0 - 3), x0 + 4)
        caja = np.nan_to_num(blanco[sy, sx])
        yy, xx = np.mgrid[sy.start:sy.start + caja.shape[0],
                          sx.start:sx.start + caja.shape[1]]
        cy = float((yy * caja).sum() / caja.sum())
        cx = float((xx * caja).sum() / caja.sum())
        ny, nx = hdu.shape[1:]
        a, b = max(0, int(cy) - half), min(ny, int(cy) + half + 1)
        c, d = max(0, int(cx) - half), min(nx, int(cx) + half + 1)
        ventana = np.asarray(hdu.data[:, a:b, c:d], dtype=np.float64)
        wave = wavelength_axis_from_header(hdu.header, int(hdu.shape[0]))
    yx = (cy - a, cx - c)
    espectros = {float(r): extract_aperture_spectrum(ventana, yx, float(r)) for r in radios}
    # FWHM sobre la imagen blanca de la ventana: perfil radial hasta medio pico.
    img = np.nanmedian(ventana[::paso], axis=0)
    iy, ix = np.indices(img.shape)
    rad = np.hypot(iy - yx[0], ix - yx[1])
    _r = np.arange(0, min(img.shape) // 2)
    with np.errstate(invalid='ignore'):
        perfil = np.array([np.nanmean(img[(rad >= k) & (rad < k + 1)]) for k in _r])
    _ok = np.isfinite(perfil)
    fwhm = (2 * float(np.interp(np.nanmax(perfil[_ok]) / 2,
                                perfil[_ok][::-1], _r[_ok][::-1]))
            if _ok.sum() > 3 else float('nan'))
    return wave, espectros, (cy, cx), fwhm

MEDIDAS = {}
_cubos = list(OBSERVACIONES) + ([('combinado', COMBINADO)]
                                if (COMBINADO and COMBINADO.exists()) else [])
if not _cubos:
    print('sin cubos que medir: la §9 no puede seguir.')
else:
    _cab = ('obs', 'DATE-OBS', 't[s]', 'X', 'DIMM["]', 'FWHM[mas]', 'y', 'x', 'forma')
    print('{:<11s}{:<21s}{:>7s}{:>7s}{:>9s}{:>11s}{:>8s}{:>8s}  {}'.format(*_cab))
    for _etq, _p in _cubos:
        _m = _cabecera_de(_p)
        _w, _esp, _yx, _fwhm = _primaria_de(_p)
        MEDIDAS[_etq] = {'wave': _w, 'esp': _esp, 'yx': _yx, 'fwhm_px': _fwhm, **_m}
        # 25.4 mas/px: la escala espacial de NFM, tomada del propio cubo mas abajo.
        print('{:<11s}{:<21s}{:>7s}{:>7s}{:>9s}{:>11s}{:>8.2f}{:>8.2f}  {}'.format(
            _etq, str(_m['date_obs'] or 'n/d')[:19],
            'n/d' if _m['exptime'] is None else f"{float(_m['exptime']):.0f}",
            'n/d' if _m['airmass'] is None else f"{_m['airmass']:.3f}",
            'n/d' if _m['seeing'] is None else f"{float(_m['seeing']):.2f}",
            'n/d' if not np.isfinite(_fwhm) else f'{_fwhm * 25.4:.0f}',
            _yx[0], _yx[1], _m['forma']))
    _u = {str(v['bunit']) for v in MEDIDAS.values()}
    print('\nBUNIT:', ', '.join(sorted(_u)),
          '(si no es uno solo, los niveles NO son comparables)' if len(_u) > 1 else '')

    # De qué OB es cada observación. Importa porque el combinado puede
    # promediar OB distintos —noches distintas— y entonces «el combinado no
    # se parece a las observaciones» tiene una explicación antes que la física.
    _obs_ob = {e: (MEDIDAS[e].get('ob_id'), MEDIDAS[e].get('ob_name'))
               for e in MEDIDAS if e != 'combinado'}
    _ob_set = {v for v in _obs_ob.values() if v[0] is not None}
    if _ob_set:
        print('\nOB de las observaciones disponibles:')
        for _id, _nom in sorted(_ob_set, key=lambda t: str(t[0])):
            _cuales = [e for e, v in _obs_ob.items() if v == (_id, _nom)]
            print(f'   OB {_id} · {_nom or "sin nombre"} → {len(_cuales)}'
                  f' exposiciones: {", ".join(_cuales)}')
        if len(_ob_set) == 1 and N_EN_COMBINADO and N_EN_COMBINADO > len(_obs_ob):
            print('   AVISO: todas las observaciones disponibles son de UN SOLO OB,')
            print(f'   pero el combinado promedia {N_EN_COMBINADO} exposiciones. Si el')
            print('   combinado no se parece a estas, mira ahí antes que a la física:')
            print('   OB distintos son noches distintas (ver docs/2026-08-06_'
                  'indicaciones_reduccion.md).')
    else:
        print('\nlas observaciones no declaran OB (`ESO OBS ID`): no se puede agrupar')


### 9.2 · Los espectros, uno por observación

El espectro de la primaria en cada exposición, con la **misma apertura** (`RADIO_PX`) y sobre su propia posición. Coloreados por masa de aire, con el combinado en rojo por encima.

**Arriba, tal cual salen.** El abanico que se ve aquí es sobre todo **instrumento**: flujo dentro de un radio fijo con una PSF que cambia. No es la estrella.

**Abajo, normalizados** a su propia mediana en la banda limpia (7750–7860 Å, sin telúrico ni líneas). Ahí ya se compara **forma con forma**, que es lo que puede ser físico: pendiente (masa de aire, extinción diferencial) y líneas.


In [ ]:
if not MEDIDAS:
    print('sin medidas: nada que dibujar')
else:
    _exps = [e for e in MEDIDAS if e != 'combinado']
    _X = {e: MEDIDAS[e]['airmass'] for e in _exps}
    _fin = [v for v in _X.values() if v]
    _norma = plt.Normalize(min(_fin), max(_fin)) if len(_fin) > 1 else None

    def _color(e):
        return plt.cm.viridis(_norma(_X[e])) if (_norma and _X[e]) else '0.6'

    def _nivel(w, s, banda=BANDA_LIMPIA_A):
        m = (w >= banda[0]) & (w <= banda[1]) & np.isfinite(s)
        return float(np.nanmedian(s[m])) if m.any() else float('nan')

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    for _a in (a1, a2):
        for _n, (_lo, _hi) in TELLURIC_BANDS.items():
            _a.axvspan(_lo, _hi, color='tab:orange', alpha=0.10, zorder=0)
        _a.axvline(6562.8, color='tab:red', lw=0.7, ls=':', zorder=1)
    for e in _exps:
        _w = MEDIDAS[e]['wave']; _s = MEDIDAS[e]['esp'][RADIO_PX]
        a1.plot(_w, _s, lw=0.5, color=_color(e), alpha=0.9)
        a2.plot(_w, _s / _nivel(_w, _s), lw=0.5, color=_color(e), alpha=0.9)
    if 'combinado' in MEDIDAS:
        _w = MEDIDAS['combinado']['wave']; _s = MEDIDAS['combinado']['esp'][RADIO_PX]
        a1.plot(_w, _s, lw=1.1, color='crimson', label='combinado')
        a2.plot(_w, _s / _nivel(_w, _s), lw=1.1, color='crimson', label='combinado')
        a1.legend(fontsize=8)
    a1.set_ylabel(f'flujo en r={RADIO_PX:.0f} px', fontsize=8)
    a1.set_title(f'la primaria en cada observación, apertura r={RADIO_PX:.0f} px '
                 '(color = masa de aire)', fontsize=9)
    a2.set_ylabel('normalizado a la banda limpia', fontsize=8)
    a2.set_xlabel('λ [Å]', fontsize=8)
    a2.axvspan(*BANDA_LIMPIA_A, color='tab:green', alpha=0.12, zorder=0)
    if _norma is not None:
        fig.colorbar(plt.cm.ScalarMappable(norm=_norma, cmap='viridis'),
                     ax=(a1, a2), label='masa de aire')
    plt.show()

    print(f'{"obs":<11s}{"nivel r=%.0f" % RADIO_PX:>13s}{"rel. mediana":>14s}')
    _niv = {e: _nivel(MEDIDAS[e]['wave'], MEDIDAS[e]['esp'][RADIO_PX]) for e in MEDIDAS}
    # La referencia es la mediana ENTRE OBSERVACIONES. Sin observaciones no
    # hay referencia y la columna se deja vacía en vez de imprimir `nan×`.
    _med = float(np.nanmedian([_niv[e] for e in _exps])) if _exps else float('nan')
    for e in list(_exps) + (['combinado'] if 'combinado' in MEDIDAS else []):
        _rel = f'{_niv[e] / _med:.2f}×' if np.isfinite(_med) else 'n/d'
        print(f'{e:<11s}{_niv[e]:>13.4g}{_rel:>14s}')
    print('\n«rel. mediana» ≠ 1 no dice nada de la estrella por sí solo: dice cuánta')
    print('luz cayó dentro del radio fijo. La §9.3 separa las dos cosas.')


### 9.3 · Por qué el nivel no es comparable: la curva de crecimiento

Misma estrella, misma apertura, distinto flujo. La explicación no está en la estrella sino en **cuánto se concentra su luz**: con una PSF más ancha, la misma cantidad de fotones cae fuera del radio.

La curva de crecimiento lo mide directamente — flujo encerrado en función del radio, normalizado al radio mayor. Si las curvas de las observaciones **no se superponen**, el nivel dentro de una apertura fija no es una medida de brillo: es una medida de AO.

Y la consecuencia práctica: para comparar **niveles** entre observaciones hay que ir a un radio grande, donde la curva ya ha saturado y da igual la concentración. Para comparar **formas**, basta con normalizar (§9.2). La tabla da la fracción encerrada en `RADIO_PX` respecto al radio mayor, que es el factor exacto por el que se está midiendo de menos en cada una.


In [ ]:
if not MEDIDAS:
    print('sin medidas: nada que dibujar')
else:
    _radios = sorted(RADIOS_PX)
    _rmax = _radios[-1]

    def _crecimiento(e):
        w = MEDIDAS[e]['wave']
        m = (w >= BANDA_LIMPIA_A[0]) & (w <= BANDA_LIMPIA_A[1])
        vals = np.array([np.nanmedian(MEDIDAS[e]['esp'][r][m]) for r in _radios])
        return vals

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    _exps = [e for e in MEDIDAS if e != 'combinado']
    _frac = {}
    for e in list(MEDIDAS):
        v = _crecimiento(e)
        _frac[e] = v / v[-1]
        _es_comb = (e == 'combinado')
        a1.plot(_radios, v, marker='o', ms=3,
                lw=1.4 if _es_comb else 0.8,
                color='crimson' if _es_comb else '0.55',
                label='combinado' if _es_comb else None)
        a2.plot(_radios, _frac[e], marker='o', ms=3,
                lw=1.4 if _es_comb else 0.8,
                color='crimson' if _es_comb else '0.55',
                label='combinado' if _es_comb else None)
    a1.set_xlabel('radio [px]', fontsize=8); a1.set_ylabel('flujo encerrado', fontsize=8)
    a1.set_yscale('log'); a1.set_title('flujo encerrado (banda limpia)', fontsize=9)
    a2.axvline(RADIO_PX, color='tab:blue', ls=':', lw=1.0,
               label=f'RADIO_PX = {RADIO_PX:.0f}')
    a2.set_xlabel('radio [px]', fontsize=8)
    a2.set_ylabel(f'fracción del flujo en r={_rmax:.0f} px', fontsize=8)
    a2.set_title('curvas de crecimiento normalizadas', fontsize=9)
    a2.legend(fontsize=7)
    for _a in (a1, a2):
        _a.tick_params(labelsize=7)
    fig.tight_layout(); plt.show()

    _i = _radios.index(RADIO_PX) if RADIO_PX in _radios else None
    print(f'{"obs":<11s}{"FWHM[mas]":>11s}{"encerrado en r=%.0f" % RADIO_PX:>20s}'
          f'{"nivel en r=%.0f" % _rmax:>17s}')
    _grande = {e: float(np.nanmedian(MEDIDAS[e]['esp'][_rmax][
        (MEDIDAS[e]['wave'] >= BANDA_LIMPIA_A[0]) &
        (MEDIDAS[e]['wave'] <= BANDA_LIMPIA_A[1])])) for e in MEDIDAS}
    _ref = float(np.nanmedian([_grande[e] for e in _exps])) if _exps else float('nan')
    for e in list(_exps) + (['combinado'] if 'combinado' in MEDIDAS else []):
        _f = MEDIDAS[e]['fwhm_px']
        _rel = f'{_grande[e] / _ref:.2f}×' if np.isfinite(_ref) else 'n/d'
        print(f'{e:<11s}{_f * 25.4 if np.isfinite(_f) else float("nan"):>11.0f}'
              f'{100 * _frac[e][_i] if _i is not None else float("nan"):>19.1f}%'
              f'{_rel:>17s}')
    print('\nSi la última columna se acerca a 1 y la anterior NO, el abanico de §9.2')
    print('era concentración de la PSF y no brillo. Si las dos se mueven, hay algo más.')


### 9.4 · ¿Se parecen las formas? El abanico, en números

Ya normalizadas, la pregunta es si las observaciones dicen lo mismo. Se mide la **dispersión entre exposiciones canal a canal** y se compara con el **ruido de cada espectro** (estimado de la propia dispersión canal a canal en la banda limpia, que en una estrella brillante es esencialmente ruido).

- **dispersión ≈ ruido** → las observaciones son la misma estrella y el combinado las representa.
- **dispersión ≫ ruido** → algo cambia entre exposiciones. Puede ser atmósfera (mira si crece con la masa de aire y dentro de las bandas telúricas: eso lo trata [`A3_telluric_debug`](A3_telluric_debug.ipynb)), o puede ser la estrella (§9.6).

La figura de abajo lo enseña por tramos de λ, que es donde se ve si la discrepancia está en el azul (extinción, respuesta) o en las bandas (telúrico).


In [ ]:
_exps = [e for e in MEDIDAS if e != 'combinado']
if len(_exps) < 2:
    print('hacen falta al menos dos observaciones para medir dispersión entre ellas.')
else:
    _w0 = MEDIDAS[_exps[0]]['wave']
    _malla_ok = all(np.allclose(MEDIDAS[e]['wave'], _w0) for e in _exps)
    print('malla de λ común a todas las observaciones:', _malla_ok)
    if not _malla_ok:
        print('   (sin malla común, comparar canal a canal mezclaría canales distintos)')
    else:
        def _norm(e):
            s = MEDIDAS[e]['esp'][RADIO_PX]
            m = (_w0 >= BANDA_LIMPIA_A[0]) & (_w0 <= BANDA_LIMPIA_A[1]) & np.isfinite(s)
            return s / float(np.nanmedian(s[m]))

        PILA = np.vstack([_norm(e) for e in _exps])
        DISP = np.nanstd(PILA, axis=0)
        # Ruido por espectro: dispersion canal a canal en la banda limpia tras
        # quitarle la tendencia (mediana movil). En una estrella brillante y sin
        # lineas, lo que queda ES el ruido.
        _lim = (_w0 >= BANDA_LIMPIA_A[0]) & (_w0 <= BANDA_LIMPIA_A[1])
        _ruidos = [float(np.nanstd((f - median_filter_1d(f, 11))[_lim])) for f in PILA]
        RUIDO = float(np.nanmedian(_ruidos)) / np.sqrt(len(_exps))
        print(f'\nruido esperado de la media de {len(_exps)} observaciones: {RUIDO:.4f}')
        print(f'{"tramo [Å]":<16s}{"dispersión":>12s}{"/ruido":>9s}   qué hay ahí')
        _tramos = [((4800, 5500), 'azul'), ((5500, 6400), 'verde'),
                   ((6400, 6700), 'Hα'), ((6700, 7500), 'rojo'),
                   (BANDA_LIMPIA_A, 'banda limpia'),
                   ((7580, 7700), 'O₂ A (telúrico)'), ((8100, 8400), 'H₂O (telúrico)')]
        for (_lo, _hi), _q in _tramos:
            _m = (_w0 >= _lo) & (_w0 <= _hi)
            if not _m.any():
                continue
            _d = float(np.nanmedian(DISP[_m]))
            print(f'{f"{_lo:.0f}–{_hi:.0f}":<16s}{_d:>12.4f}{_d / RUIDO:>9.1f}×   {_q}')

        fig, ax = plt.subplots(figsize=(12, 3.4))
        for _n, (_lo, _hi) in TELLURIC_BANDS.items():
            ax.axvspan(_lo, _hi, color='tab:orange', alpha=0.12, zorder=0)
        ax.plot(_w0, DISP, lw=0.6, color='tab:purple', label='dispersión entre observaciones')
        ax.axhline(RUIDO, color='tab:blue', ls='--', lw=1.0,
                   label='ruido esperado de la media')
        ax.set_yscale('log'); ax.legend(fontsize=7)
        ax.set_xlabel('λ [Å]', fontsize=8)
        ax.set_ylabel('σ entre observaciones\n(normalizadas)', fontsize=8)
        ax.set_title('cuánto se separan las observaciones, canal a canal', fontsize=9)
        fig.tight_layout(); plt.show()


### 9.5 · ¿Suman al combinado?

La prueba de cierre: promediar las observaciones —pesando por `EXPTIME`, que es lo que hace el DRS— y comparar con la primaria medida **en el cubo combinado**. En **forma** deben coincidir; el **nivel** solo coincide si los cubos por exposición son de verdad los que entraron en el combinado y en la misma unidad, así que el cociente de niveles se imprime aparte y se interpreta con lo que dijo la §9.

> Ojo: **sumar espectros no es combinar cubos.** El DRS combina píxel a píxel con sus pesos y su remuestreo; esto promedia espectros ya extraídos. Sirve para comparar **las dos formas**, no para reproducir la combinación.


In [ ]:
_exps = [e for e in MEDIDAS if e != 'combinado']
if 'combinado' not in MEDIDAS or not _exps:
    print('hace falta el combinado y al menos una observación.')
elif not all(np.allclose(MEDIDAS[e]['wave'], MEDIDAS['combinado']['wave']) for e in _exps):
    print('las mallas de λ no coinciden: promediar mezclaría canales distintos.')
else:
    _rmax = float(max(RADIOS_PX))
    _w = MEDIDAS['combinado']['wave']
    _peso = {e: float(MEDIDAS[e]['exptime'] or 0.0) for e in _exps}
    if sum(_peso.values()) <= 0:
        print('ninguna exposición declara EXPTIME: se promedia sin pesos.')
        _peso = {e: 1.0 for e in _exps}
    for _r, _nombre in ((RADIO_PX, f'apertura de trabajo r={RADIO_PX:.0f} px'),
                        (_rmax, f'radio grande r={_rmax:.0f} px')):
        _suma = sum(MEDIDAS[e]['esp'][_r] * _peso[e] for e in _exps) / sum(_peso.values())
        _comb = MEDIDAS['combinado']['esp'][_r]
        _m = (_w >= BANDA_LIMPIA_A[0]) & (_w <= BANDA_LIMPIA_A[1])
        _razon = float(np.nanmedian(_comb[_m]) / np.nanmedian(_suma[_m]))
        # La FORMA: cociente de los dos, ya normalizados cada uno a su nivel.
        with np.errstate(invalid='ignore', divide='ignore'):
            _forma = (_comb / np.nanmedian(_comb[_m])) / (_suma / np.nanmedian(_suma[_m]))
        _fin = np.isfinite(_forma) & (_w > 5000) & (_w < 9000)
        print(f'{_nombre}:')
        print(f'   nivel combinado / media pesada = {_razon:.3f}×')
        print(f'   forma: mediana {np.nanmedian(_forma[_fin]):.4f}, '
              f'p5..p95 {np.nanpercentile(_forma[_fin], 5):.4f}..'
              f'{np.nanpercentile(_forma[_fin], 95):.4f}  (1 = misma forma)')

    _suma = sum(MEDIDAS[e]['esp'][_rmax] * _peso[e] for e in _exps) / sum(_peso.values())
    _comb = MEDIDAS['combinado']['esp'][_rmax]
    _m = (_w >= BANDA_LIMPIA_A[0]) & (_w <= BANDA_LIMPIA_A[1])
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 5.4), sharex=True,
                                 gridspec_kw={'height_ratios': [2, 1]})
    for _n, (_lo, _hi) in TELLURIC_BANDS.items():
        for _a in (a1, a2):
            _a.axvspan(_lo, _hi, color='tab:orange', alpha=0.10, zorder=0)
    a1.plot(_w, _suma / np.nanmedian(_suma[_m]), lw=0.6, color='tab:blue',
            label=f'media pesada de {len(_exps)} observaciones')
    a1.plot(_w, _comb / np.nanmedian(_comb[_m]), lw=0.6, color='crimson',
            label='medida en el cubo combinado')
    a1.legend(fontsize=8); a1.set_ylabel('normalizado', fontsize=8)
    a1.set_title(f'la suma de las observaciones contra el combinado (r={_rmax:.0f} px)',
                 fontsize=9)
    with np.errstate(invalid='ignore', divide='ignore'):
        a2.plot(_w, (_comb / np.nanmedian(_comb[_m])) / (_suma / np.nanmedian(_suma[_m])),
                lw=0.6, color='tab:purple')
    a2.axhline(1.0, color='0.6', lw=0.8, ls=':')
    a2.set_ylim(0.7, 1.3)
    a2.set_ylabel('combinado / suma', fontsize=8); a2.set_xlabel('λ [Å]', fontsize=8)
    fig.tight_layout(); plt.show()


### 9.6 · Hα de la primaria, observación a observación

La única de estas medidas que puede ser **de la estrella**. La primaria de un sistema así es una fuente de acreción joven y su Hα varía en horas: si el flujo de línea cambia entre exposiciones más de lo que cambia el continuo, eso **no** es AO ni atmósfera.

Se mide sobre el espectro normalizado a su continuo local (bandas laterales a los dos lados de la línea, fuera de ella), así que la variación de la PSF —que multiplica todo el espectro por igual— se cancela por construcción. Lo que queda es **anchura equivalente**: emisión en negativo, por convención.

**Antes de leer nada como variabilidad, la prueba de la apertura.** La tabla mide el EW a **todos los radios**, y ese es el discriminante:

- **EW plano con el radio** → la línea y el continuo tienen el mismo perfil espacial: la medida es de la fuente puntual y no depende de cuánta PSF entre en la apertura. Entonces una diferencia entre observaciones **es** una diferencia entre los datos, no un efecto de AO.
- **EW que crece con el radio** → hay emisión **extendida** (nebular, del campo) sumándose a la de la estrella, y comparar aperturas distintas no compara lo mismo.

> Esto es un diagnóstico de la **primaria**, no del compañero. No entra en ningún límite de acreción del objeto (eso es el bloque E) y no puede sustituir a una curva de luz: unos pocos puntos en unas horas dicen si algo se mueve, no cuánto.

**Si el combinado cae fuera del rango de las observaciones** y el EW es plano con el radio, no lo explica ni la apertura ni la PSF: quedan dos causas, y la §9.1 dice cuál mirar primero. Si el combinado **mezcla varios OB** y las observaciones disponibles son de uno solo, lo que se está viendo es la diferencia entre OB promediada — que es exactamente el motivo de la decisión de [`docs/2026-08-06_indicaciones_reduccion.md`](../../../docs/2026-08-06_indicaciones_reduccion.md) (un OB, un cubo). La otra causa sería la propia combinación, y para separarlas hace falta el cubo de cada OB por separado.


In [ ]:
def _ew_halpha(w, s):
    """(flujo de linea, EW) sobre el continuo local. EW<0 = emision."""
    lat = np.zeros(w.shape, dtype=bool)
    for _lo, _hi in HALPHA_LATERAL_A:
        lat |= (w >= _lo) & (w <= _hi)
    nuc = (w >= HALPHA_A[0]) & (w <= HALPHA_A[1])
    ok = lat & np.isfinite(s)
    if ok.sum() < 10 or not nuc.any():
        return float('nan'), float('nan')
    cont = np.polyval(np.polyfit(w[ok], s[ok], 1), w)
    dl = float(np.median(np.diff(w)))
    exceso = (s - cont)[nuc]
    with np.errstate(invalid='ignore', divide='ignore'):
        ew = -float(np.nansum(exceso / cont[nuc]) * dl)
    return float(np.nansum(exceso) * dl), ew

if not MEDIDAS:
    print('sin medidas: nada que medir en Hα')
else:
    _exps = [e for e in MEDIDAS if e != 'combinado']
    _orden = _exps + (['combinado'] if 'combinado' in MEDIDAS else [])
    print(f'{"obs":<11s}{"X":>7s}{"EW(Hα) [Å]":>13s}{"flujo línea":>14s}')
    _ews = {}
    for e in _orden:
        _w = MEDIDAS[e]['wave']; _s = MEDIDAS[e]['esp'][RADIO_PX]
        _f, _ew = _ew_halpha(_w, _s)
        _ews[e] = _ew
        _x = MEDIDAS[e]['airmass']
        print(f'{e:<11s}{("n/d" if _x is None else f"{_x:.3f}"):>7s}'
              f'{_ew:>13.3f}{_f:>14.4g}')
    # LA PRUEBA DE LA APERTURA. Si el EW no depende del radio, la linea y el
    # continuo tienen el mismo perfil espacial y una diferencia entre
    # observaciones NO puede explicarse por la PSF ni por emision extendida.
    _radios = sorted(RADIOS_PX)
    print('\nEW(Hα) en función del radio de apertura:')
    print('obs        ' + ''.join(f'{("r=" + str(int(_r))):>9s}' for _r in _radios)
          + f'{"variación":>12s}')
    VARIACION_MAX_PCT = 25.0   # por encima, la medida no es de fiar
    _varia = {}
    for e in _orden:
        _fila = [_ew_halpha(MEDIDAS[e]['wave'], MEDIDAS[e]['esp'][_r])[1] for _r in _radios]
        _f = np.array(_fila, dtype=float)
        _var = (100 * (np.nanmax(_f) - np.nanmin(_f)) / abs(np.nanmedian(_f))
                if np.isfinite(_f).any() else float('nan'))
        _varia[e] = _var
        _marca = '  <-- inestable' if _var > VARIACION_MAX_PCT else ''
        print(f'{e:<11s}' + ''.join(f'{v:>9.2f}' for v in _fila) + f'{_var:>11.0f}%{_marca}')
    print('«variación» pequeña = EW plano con el radio = fuente puntual, y entonces')
    print('las diferencias entre observaciones no las explica la apertura ni la PSF.')
    _malas = [e for e in _exps if _varia.get(e, 0) > VARIACION_MAX_PCT]
    if _malas:
        print(f'\n{len(_malas)} observación(es) con EW que SÍ depende del radio: '
              + ', '.join(_malas))
        print('En ellas la línea y el continuo no comparten perfil espacial, así que su')
        print('EW no mide lo mismo que en las demás. Quedan fuera del resumen de abajo,')
        print('y por qué pasa en esa exposición y no en sus vecinas es una pregunta viva.')

    # El resumen va POR OB, que es el agrupamiento de §9.1: promediar dos
    # OB en una sola dispersión mezcla dos noches y no mide variabilidad
    # de nada. Y con mediana/MAD, para que una observación mala no mande.
    _ob_de = globals().get('_obs_ob') or {}
    _grupos = {}
    for e in _exps:
        if e in _malas or not np.isfinite(_ews.get(e, np.nan)):
            continue
        _grupos.setdefault(_ob_de.get(e, (None, None)), []).append(e)
    if _grupos:
        print(f'\n{"OB":<22s}{"n":>4s}{"EW mediana":>13s}{"MAD":>9s}{"min..max":>18s}')
        for (_id, _nom), _es in sorted(_grupos.items(), key=lambda t: str(t[0][1])):
            _v = np.array([_ews[e] for e in _es], dtype=float)
            _mad = float(np.median(np.abs(_v - np.median(_v))))
            _etq = f'{_nom or "sin OB"}'
            print(f'{_etq:<22s}{len(_v):>4d}{np.median(_v):>13.3f}{_mad:>9.3f}'
                  f'{f"{_v.min():.2f}..{_v.max():.2f}":>18s}')
        if len(_grupos) > 1:
            _meds = [float(np.median([_ews[e] for e in _es])) for _es in _grupos.values()]
            print(f'\nentre OB: la mediana va de {min(_meds):+.2f} a {max(_meds):+.2f} Å '
                  f'(factor {max(_meds, key=abs) / min(_meds, key=abs):.1f}).')
            print('La dispersión DENTRO de cada OB (la MAD) es la referencia contra la que')
            print('juzgar esa diferencia: si es mucho menor, los dos OB no son el mismo dato,')
            print('y promediarlos en un solo cubo es una decisión, no un detalle.')
        else:
            print('\nun solo OB: la dispersión de arriba es el límite de lo que se puede')
            print('decir sobre variabilidad; para afirmarla hace falta el error de cada EW,')
            print('que estos datos no dan por separado.')

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8),
                                 gridspec_kw={'width_ratios': [2, 1]})
    for e in _orden:
        _w = MEDIDAS[e]['wave']; _s = MEDIDAS[e]['esp'][RADIO_PX]
        _lat = np.zeros(_w.shape, dtype=bool)
        for _lo, _hi in HALPHA_LATERAL_A:
            _lat |= (_w >= _lo) & (_w <= _hi)
        _ok = _lat & np.isfinite(_s)
        if _ok.sum() < 10:
            continue
        _cont = np.polyval(np.polyfit(_w[_ok], _s[_ok], 1), _w)
        _sel = (_w >= 6450) & (_w <= 6680)
        _comb = (e == 'combinado')
        a1.plot(_w[_sel], (_s / _cont)[_sel], lw=1.2 if _comb else 0.7,
                color='crimson' if _comb else '0.55',
                label='combinado' if _comb else None, zorder=3 if _comb else 2)
    a1.axvspan(*HALPHA_A, color='tab:red', alpha=0.10)
    a1.axhline(1.0, color='0.6', lw=0.8, ls=':')
    a1.set_xlabel('λ [Å]', fontsize=8); a1.set_ylabel('flujo / continuo local', fontsize=8)
    a1.set_title('Hα de la primaria en cada observación', fontsize=9)
    a1.legend(fontsize=7)
    _t = [MEDIDAS[e]['airmass'] for e in _exps]
    if any(v for v in _t):
        a2.plot([v for v in _t], [_ews[e] for e in _exps], 'o', ms=5, color='tab:blue')
        a2.set_xlabel('masa de aire', fontsize=8)
    else:
        a2.plot(range(len(_exps)), [_ews[e] for e in _exps], 'o', ms=5, color='tab:blue')
        a2.set_xlabel('observación', fontsize=8)
    if 'combinado' in _ews and np.isfinite(_ews['combinado']):
        a2.axhline(_ews['combinado'], color='crimson', lw=1.2, label='combinado')
        a2.legend(fontsize=7)
    a2.set_ylabel('EW(Hα) [Å]', fontsize=8)
    a2.set_title('¿depende de la atmósfera?', fontsize=9)
    for _a in (a1, a2):
        _a.tick_params(labelsize=7)
    fig.tight_layout(); plt.show()


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'psffit_star_calibrated'
    PRODUCT_P = 'spec_calibrated_psffit_star.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/stage_x11_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'la PRIMARIA calibrada (producto de D2)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'd2_primaria'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_primaria' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_primaria' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_primaria' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## 10 · ¿Es correcta la FORMA? El cociente contra el cubo

La figura de arriba es el producto de la cadena. Esta sección comprueba que su **pendiente** sea la del dato, con la prueba más simple que existe: medir la misma estrella por un camino que no comparte casi nada con el psffit —una **apertura circular grande directamente sobre el cubo de entrada**— y **dividir**.

Los dos miden la misma estrella, así que su cociente **tiene que ser plano** salvo por la corrección de apertura, que es cromática a propósito (con AO el Strehl empeora hacia el azul, más luz se va al halo, y una apertura fija captura menos). Por eso el cociente se compara con **`apcorr`**: si la corrección es correcta, `cociente / apcorr` sale constante.

Y hay un tercer testigo, independiente de los dos: la **curva de crecimiento del modelo de PSF de C1**, `F(R)/F(r_norm)`. Es un modelo físico ajustado al dato, y su forma tiene que ser **monótona decreciente** en λ. Si `apcorr` no lo es, la corrección está mal aunque su valor medio parezca razonable.

> **Por qué está aquí.** Este cociente destapó, el 2026-08-07, que el factor empírico `normrad → total` estaba medido sobre un cubo demasiado pequeño: halo y cielo salían degenerados, el exponente del halo colapsaba a 2.2–2.6 (Kolmogorov da 3.67), casi la mitad del «total» era cola extrapolada, y la `apcorr` resultante tenía un **máximo interior** hacia 7400 Å que deformaba la pendiente del continuo un ~47 % — en todos los métodos del bloque C, no solo en la primaria. Es el hermano del `sigma_inflation` de G2: una métrica de salud que se calcula con los mismos datos en cada corrida, y cuyo valor está en que **se mantenga plana**.


In [ ]:
from musepipe.psf import evaluate_psf_model
from musepipe.extraction.aperture import aperture_weights

RADIO_FORMA_PX = 32.0      # apertura grande: captura la mayor parte del halo
BANDA_REF_A = (7750.0, 7860.0)   # donde se normalizan las tres curvas

_cubo_b1, _ = nb.entry_cube(RUN_ID)   # el cubo por el que ENTRA la cadena hoy
if _cubo_b1 is None or not Path(_cubo_b1).exists():
    print('sin cubo de entrada declarado por B1: no se puede hacer el cociente')
else:
    with fits.open(_cubo_b1, memmap=True) as _h:
        _hdu = _h['DATA'] if 'DATA' in _h else _h[1]
        _b = np.nanmedian(np.asarray(_hdu.data[::PASO_BLANCO], dtype=np.float64), axis=0)
        _y0, _x0 = (int(v) for v in np.unravel_index(np.nanargmax(_b), _b.shape))
        _ny, _nx = _hdu.shape[1:]
        _a, _bb = max(0, _y0 - 45), min(_ny, _y0 + 46)
        _c, _d = max(0, _x0 - 45), min(_nx, _x0 + 46)
        _v = np.asarray(_hdu.data[:, _a:_bb, _c:_d], dtype=np.float64)
        _w_cubo = wavelength_axis_from_header(_hdu.header, int(_hdu.shape[0]))
    F_APERTURA = extract_aperture_spectrum(_v, (_y0 - _a, _x0 - _c), RADIO_FORMA_PX)
    F_APERTURA = np.interp(WAVE, _w_cubo, F_APERTURA)   # a la malla del producto

    def _norm(w, f, banda=BANDA_REF_A):
        m = (w >= banda[0]) & (w <= banda[1]) & np.isfinite(f)
        return f / float(np.nanmedian(f[m])) if m.any() else f * np.nan

    with np.errstate(invalid='ignore', divide='ignore'):
        COCIENTE = FLUJO / F_APERTURA
    APC = np.asarray(ESTRELLA.apcorr, dtype=np.float64)
    # Tercer testigo: la curva del modelo de C1, calculada aquí mismo.
    _psf_path = SD / 'psf_model.json'
    MODELO = None
    if _psf_path.exists():
        _m = json.loads(_psf_path.read_text(encoding='utf-8'))
        _rn = float(_m.get('norm_radius_px', 25.0)); _R = int(RADIO_FORMA_PX) + 8
        _yy, _xx = np.indices((2 * _R + 1, 2 * _R + 1), dtype=np.float64)
        _dy, _dx = _yy - _R, _xx - _R
        _wn = aperture_weights(2 * _R + 1, 2 * _R + 1, (_R, _R),
                               {'kind': 'circle', 'radius_px': _rn})
        _wg = aperture_weights(2 * _R + 1, 2 * _R + 1, (_R, _R),
                               {'kind': 'circle', 'radius_px': float(_R)})
        _lam = np.linspace(WAVE[0], WAVE[-1], 40)
        _rat = []
        for _l in _lam:
            _p = evaluate_psf_model(_m, float(_l), _dy, _dx)
            _rat.append(float(np.nansum(_p * _wg)) / float(np.nansum(_p * _wn)))
        MODELO = np.interp(WAVE, _lam, _rat)

    _fin = np.isfinite(COCIENTE) & (WAVE > 4800) & (WAVE < 9300)
    _resid = _norm(WAVE, COCIENTE) / _norm(WAVE, APC)
    _pp = float(np.nanpercentile(_resid[_fin], 98) / np.nanpercentile(_resid[_fin], 2))
    print(f'cociente producto/apertura: {np.nanpercentile(COCIENTE[_fin], 2):.3f}'
          f' .. {np.nanpercentile(COCIENTE[_fin], 98):.3f}'
          f'  (factor {np.nanpercentile(COCIENTE[_fin], 98) / np.nanpercentile(COCIENTE[_fin], 2):.2f})')
    print(f'tras dividir por apcorr:    varía un factor {_pp:.3f}'
          f"   -> {'PLANO, la corrección explica la forma' if _pp < 1.10 else 'NO PLANO: la apcorr no explica la forma'}")
    # `interior_bump`, no el signo de la derivada: un repunte del 1% en el
    # borde por extrapolación del ajuste no es una joroba, y un test
    # estricto lo marcaba igual que el máximo interior del 16% que sí lo era.
    from musepipe.growth_curve import interior_bump, MAX_INTERIOR_BUMP
    _joroba = interior_bump(APC)
    _mono = _joroba <= MAX_INTERIOR_BUMP
    print(f'apcorr monótona en λ:       {"sí" if _mono else "NO"}  (joroba {100 * _joroba:.2f}%)')
    if MODELO is not None:
        _jm = interior_bump(MODELO)
        print(f'modelo de C1 monótono:      {"sí" if _jm <= MAX_INTERIOR_BUMP else "NO"}  (joroba {100 * _jm:.2f}%)')
    if not _mono:
        print('\nLa apcorr tiene una joroba: la curva de crecimiento está mal medida.')
        print('Mira `stages/growth_curve_qc.json`: halo_power (físico 2.5-4),')
        print('tail_fraction (<=0.25) y la dispersión ratio_max/ratio_min por banda.')
        print('Se re-mide con measure_growth_curve.py sobre un combinado más ancho.')
    elif _pp >= 1.10:
        print(f'\nLa apcorr es sana (monótona) pero AÚN queda un {100 * (_pp - 1):.0f}% de')
        print('forma sin explicar. Eso ya no es la curva de crecimiento. Lo que queda')
        print('entre las dos medidas es cuánto se parece el MODELO de PSF de C1 al')
        print('perfil real en función de λ: el psffit devuelve la amplitud de un')
        print('modelo ajustado, la apertura suma píxeles. Contrástalo con el residuo')
        print('del ajuste de C1 (`psf_hybrid_residual.fits`) y con `psf_roundtrip_error`.')

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11.5, 6), sharex=True,
                                 gridspec_kw={'height_ratios': [1.4, 1]})
    a1.plot(WAVE, _norm(WAVE, COCIENTE), lw=0.8, color='tab:blue',
            label='producto / apertura sobre el cubo')
    a1.plot(WAVE, _norm(WAVE, APC), lw=1.4, color='tab:red', label='apcorr de la cadena')
    if MODELO is not None:
        a1.plot(WAVE, _norm(WAVE, MODELO), lw=1.4, color='tab:green', ls='--',
                label='curva del modelo de PSF (C1)')
    a1.axvspan(*BANDA_REF_A, color='0.85', zorder=0)
    a1.legend(fontsize=8); a1.set_ylabel('normalizado a la banda de referencia', fontsize=8)
    a1.set_title('la forma, por tres caminos: si los tres coinciden, la pendiente es del dato',
                 fontsize=9)
    a2.plot(WAVE, _resid, lw=0.8, color='tab:purple')
    a2.axhline(1.0, color='0.6', lw=0.8, ls=':')
    a2.set_ylim(0.6, 1.4)
    a2.set_ylabel('cociente / apcorr\n(1 = la apcorr lo explica)', fontsize=8)
    a2.set_xlabel('λ [Å]', fontsize=8)
    fig.tight_layout(); plt.show()


## 11 · Qué NO decide este notebook

1. **No cambia la cadena.** Lo que se toque aquí no toca `musepipe`; si un cambio merece entrar, entra por la etapa y con su aprobación.
2. **No mide variabilidad de la primaria con error.** La §9.6 da la dispersión entre observaciones, que es una **cota** de lo que se puede afirmar: sin una barra de error por EW, «se mueve» no está demostrado.
3. **No dice que los cubos por exposición sean los del combinado.** Lo comprueba y lo avisa, pero si no coinciden, la §9 describe esas observaciones y nada más — y entonces la §9.5 no es una prueba de cierre sino una comparación entre dos reducciones distintas.
4. **No separa telúrico de fotosférico.** Un rasgo común a todas las observaciones no es por eso estelar: lo común incluye todo lo que no depende de la masa de aire. Esa pregunta la trabaja [`A3_telluric_debug`](A3_telluric_debug.ipynb) §14.5.
5. **No toca el compañero.** El contraste, los límites de Hα y la acreción del objeto son de los bloques D1/E, no de aquí.
